# Seed Inventra (J&F)

Carga de demonstração do banco do Inventra, ambientada nas cozinhas industriais das empresas de alimentos do grupo J&F (JBS, Friboi, Seara e Swift).

- **500 linhas por tabela** em todas as tabelas de cadastro e movimento (usuários, produtos, fornecedores, lotes, movimentações, requisições, inventários, alertas etc.).
- As tabelas de domínio ficam com o tamanho real do negócio: 3 perfis de acesso, 7 categorias oficiais, as unidades de medida usadas no recebimento e 40 cozinhas.
- Nada de "Produto Genérico 67" ou lorem ipsum: produtos, embalagens, preços, validades, motivos e mensagens seguem o que uma cozinha industrial realmente registra, e os dados conversam entre si (o saldo de cada lote fecha com as movimentações, alertas de estoque mínimo só existem onde o estoque está abaixo do mínimo, e assim por diante).

# IMPORTS E CONFIGURAÇÃO INICIAL

In [ ]:
from faker import Faker
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import random
import re
import unicodedata
import os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, date, time, timedelta
from pathlib import Path
import json
import time as clock
import urllib.parse
import urllib.request

fake = Faker("pt_BR")
load_dotenv()
random.seed(42)
Faker.seed(42)

# Quantidade de linhas das tabelas de cadastro e movimento.
ROWS_PER_TABLE = 500

NOW = datetime.now().replace(microsecond=0)
TODAY = NOW.date()

print("Bibliotecas carregadas")

# CONEXÃO COM O BANCO DE DADOS

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE", "require")
)
cursor = conn.cursor()

cursor.execute("SELECT current_database(), current_user")
_db, _user = cursor.fetchone()
print(f"Conectado ao PostgreSQL: banco '{_db}' como '{_user}' em {os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}")

# RESET TOTAL DO BANCO

In [ ]:
cursor.execute("""
TRUNCATE TABLE
    tb_profile,
    tb_kitchen,
    tb_category,
    tb_measurement_unit,
    tb_user,
    tb_product,
    tb_supplier,
    tb_product_supplier,
    tb_product_kitchen_parameter,
    tb_stock_batch,
    tb_stock_movement,
    tb_requisition,
    tb_requisition_item,
    tb_inventory,
    tb_inventory_count,
    tb_alert,
    tb_alert_read,
    tb_user_daily_access
RESTART IDENTITY CASCADE;
""")
cursor.execute("""
TRUNCATE TABLE
    tb_log_user,
    tb_log_product,
    tb_log_supplier,
    tb_log_stock_batch,
    tb_log_requisition,
    tb_log_inventory,
    tb_log_alert
RESTART IDENTITY;
""")
conn.commit()

print("Banco resetado com sucesso")

# DOMÍNIO: EMPRESAS, COZINHAS E HELPERS

Os dados são fictícios, ambientados nas empresas de alimentos do grupo J&F. As cozinhas ficam em cidades onde o grupo tem operação, com coordenadas da cidade; CNPJs, e-mails, telefones, códigos de barras e notas fiscais são gerados e não correspondem a registros reais.

In [ ]:
COMPANIES = {
    "JBS": {"name": "JBS", "domain": "jbs.com.br"},
    "FRIBOI": {"name": "Friboi", "domain": "friboi.com.br"},
    "SEARA": {"name": "Seara", "domain": "seara.com.br"},
    "SWIFT": {"name": "Swift", "domain": "swift.com.br"},
}

# (empresa, unidade, cidade, UF, latitude, longitude, endereço, refeições servidas por dia)
KITCHENS = [
    ("FRIBOI", "Friboi Lins", "Lins", "SP", -21.6786, -49.7425, "Rodovia Marechal Rondon (SP-300), km 437 — Distrito Industrial", 3800),
    ("FRIBOI", "Friboi Andradina", "Andradina", "SP", -20.8962, -51.3794, "Rodovia Marechal Rondon (SP-300), km 625", 2600),
    ("FRIBOI", "Friboi Barretos", "Barretos", "SP", -20.5572, -48.5678, "Rodovia Brigadeiro Faria Lima (SP-326), km 428", 2400),
    ("FRIBOI", "Friboi Campo Grande", "Campo Grande", "MS", -20.4697, -54.6201, "Rodovia BR-163, km 474 — Núcleo Industrial", 3200),
    ("FRIBOI", "Friboi Naviraí", "Naviraí", "MS", -23.0618, -54.1990, "Rodovia BR-163, km 89", 1800),
    ("FRIBOI", "Friboi Ponta Porã", "Ponta Porã", "MS", -22.5361, -55.7256, "Rodovia BR-463, km 4", 1500),
    ("FRIBOI", "Friboi Goiânia", "Goiânia", "GO", -16.6869, -49.2648, "Rodovia BR-153, km 5 — Setor Perimetral Norte", 2900),
    ("FRIBOI", "Friboi Senador Canedo", "Senador Canedo", "GO", -16.7084, -49.0914, "Rodovia GO-403, km 3 — Distrito Agroindustrial", 2100),
    ("FRIBOI", "Friboi Mozarlândia", "Mozarlândia", "GO", -14.7457, -50.5713, "Rodovia GO-164, km 2", 1400),
    ("FRIBOI", "Friboi Barra do Garças", "Barra do Garças", "MT", -15.8900, -52.2567, "Rodovia BR-070, km 6", 1700),
    ("FRIBOI", "Friboi Diamantino", "Diamantino", "MT", -14.4086, -56.4461, "Rodovia MT-240, km 3", 1300),
    ("FRIBOI", "Friboi Pontes e Lacerda", "Pontes e Lacerda", "MT", -15.2261, -59.3353, "Rodovia BR-174, km 12", 1600),
    ("FRIBOI", "Friboi Vilhena", "Vilhena", "RO", -12.7406, -60.1458, "Rodovia BR-364, km 8", 1500),
    ("FRIBOI", "Friboi Marabá", "Marabá", "PA", -5.3686, -49.1178, "Rodovia BR-155, km 11 — Distrito Industrial", 1900),
    ("FRIBOI", "Friboi Araguaína", "Araguaína", "TO", -7.1911, -48.2072, "Rodovia BR-153, km 132", 1700),
    ("FRIBOI", "Friboi Ituiutaba", "Ituiutaba", "MG", -18.9746, -49.4653, "Rodovia BR-365, km 739", 1600),
    ("SEARA", "Seara Itajaí", "Itajaí", "SC", -26.9078, -48.6619, "Rodovia Antônio Heil (SC-486), km 2 — Itaipava", 3400),
    ("SEARA", "Seara Forquilhinha", "Forquilhinha", "SC", -28.7454, -49.4720, "Rodovia SC-447, km 4", 2700),
    ("SEARA", "Seara Matriz", "Seara", "SC", -27.1564, -52.3086, "Avenida Anita Garibaldi, 2100 — Distrito Industrial", 2300),
    ("SEARA", "Seara Jaraguá do Sul", "Jaraguá do Sul", "SC", -26.4851, -49.0713, "Rodovia BR-280, km 52", 1900),
    ("SEARA", "Seara Dourados", "Dourados", "MS", -22.2231, -54.8120, "Rodovia BR-163, km 268", 2500),
    ("SEARA", "Seara Sidrolândia", "Sidrolândia", "MS", -20.9302, -54.9692, "Rodovia MS-162, km 5", 1800),
    ("SEARA", "Seara Rolândia", "Rolândia", "PR", -23.3101, -51.3659, "Rodovia BR-369, km 162", 2200),
    ("SEARA", "Seara Carambeí", "Carambeí", "PR", -24.9175, -50.0986, "Rodovia PR-151, km 318", 1600),
    ("SEARA", "Seara Nuporanga", "Nuporanga", "SP", -20.7296, -47.7428, "Rodovia Altino Arantes (SP-351), km 40", 1700),
    ("SEARA", "Seara Montenegro", "Montenegro", "RS", -29.6888, -51.4613, "Rodovia RS-124, km 6", 2000),
    ("SEARA", "Seara Passo Fundo", "Passo Fundo", "RS", -28.2620, -52.4083, "Rodovia BR-285, km 292", 2100),
    ("SEARA", "Seara Caxias do Sul", "Caxias do Sul", "RS", -29.1678, -51.1794, "Rodovia RS-122, km 71 — Ana Rech", 1500),
    ("SEARA", "Seara Rio Verde", "Rio Verde", "GO", -17.7923, -50.9192, "Rodovia BR-060, km 381", 2600),
    ("SEARA", "Seara Amparo", "Amparo", "SP", -22.7011, -46.7647, "Rodovia SP-95, km 38", 1400),
    ("SEARA", "Seara CD Jundiaí", "Jundiaí", "SP", -23.1857, -46.8978, "Rodovia Anhanguera (SP-330), km 58 — Centro de Distribuição", 900),
    ("SEARA", "Seara Lajeado", "Lajeado", "RS", -29.4669, -51.9611, "Rodovia BR-386, km 350", 1500),
    ("SWIFT", "Swift CD São Paulo", "São Paulo", "SP", -23.5205, -46.7262, "Avenida Mofarrej, 1200 — Vila Leopoldina", 700),
    ("SWIFT", "Swift CD Campinas", "Campinas", "SP", -22.8590, -47.0337, "Rodovia Dom Pedro I (SP-065), km 136", 600),
    ("SWIFT", "Swift CD Rio de Janeiro", "Rio de Janeiro", "RJ", -22.8253, -43.3127, "Avenida Brasil, 21500 — Pavuna", 650),
    ("SWIFT", "Swift CD Belo Horizonte", "Belo Horizonte", "MG", -19.8697, -43.9820, "Anel Rodoviário Celso Mello Azevedo, km 23", 550),
    ("SWIFT", "Swift CD Curitiba", "Curitiba", "PR", -25.5135, -49.2386, "Rodovia BR-116, km 105 — Cidade Industrial", 500),
    ("JBS", "JBS Sede Administrativa", "São Paulo", "SP", -23.5126, -46.6945, "Avenida Marginal Direita do Tietê, 500 — Vila Jaguara", 1200),
    ("JBS", "JBS Couros Lins", "Lins", "SP", -21.6650, -49.7180, "Rodovia Marechal Rondon (SP-300), km 440", 900),
    ("JBS", "JBS Couros Cacoal", "Cacoal", "RO", -11.4386, -61.4472, "Rodovia BR-364, km 186", 700),
]

# DDDs por UF, para o WhatsApp dos fornecedores regionais
DDD_BY_UF = {
    "SP": [11, 14, 15, 16, 17, 18, 19], "MS": [67], "GO": [62, 64], "MT": [65, 66], "RO": [69],
    "PA": [91, 94], "TO": [63], "MG": [31, 34, 35, 37, 38], "SC": [47, 48, 49], "PR": [41, 42, 43, 44],
    "RS": [51, 54, 55], "RJ": [21, 22, 24],
}

# unidades em que o estoque é fracionado (kg, litro); nas demais a contagem é inteira
MEASURE_UNITS = {"KG", "G", "L", "ML"}


def slugify(text, sep="."):
    nfkd = unicodedata.normalize("NFKD", text)
    ascii_text = nfkd.encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", sep, ascii_text).strip(sep)


def rand_datetime(start, end):
    span = max((end - start).total_seconds(), 0)
    return (start + timedelta(seconds=random.uniform(0, span))).replace(microsecond=0)


def work_datetime(day, start_hour=6, end_hour=18):
    """Horário comercial aleatório no dia informado, nunca depois de agora."""
    dt = datetime.combine(day, time(start_hour)) + timedelta(minutes=random.randint(0, (end_hour - start_hour) * 60 - 1))
    return min(dt, NOW - timedelta(minutes=random.randint(5, 90)))


def past_work_datetime(min_days, max_days, start_hour=6, end_hour=18):
    return work_datetime(TODAY - timedelta(days=random.randint(min_days, max_days)), start_hour, end_hour)


def ean13():
    """EAN-13 com prefixo GS1 Brasil (789) e dígito verificador válido."""
    body = "789" + "".join(random.choices("0123456789", k=9))
    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(body))
    return body + str((10 - total % 10) % 10)


def qty_step(unit):
    return 0.1 if unit in MEASURE_UNITS else 1


def round_qty(value, unit):
    step = qty_step(unit)
    return round(max(step, round(value / step) * step), 3)


def to_milli(value):
    return int(round(value * 1000))


def from_milli(value):
    return round(value / 1000, 3)


def split_milli(total, parts, step):
    """Divide total (em milésimos) em partes positivas múltiplas de step."""
    parts = max(1, min(parts, total // step))
    weights = [random.uniform(0.5, 1.5) for _ in range(parts)]
    out = [max(step, int(total * w / sum(weights)) // step * step) for w in weights]
    out[-1] = total - sum(out[:-1])
    while out[-1] <= 0 and len(out) > 1:
        extra = out.pop()
        out[-1] += extra
    return out


print(f"Domínio J&F carregado: {len(KITCHENS)} cozinhas em {len({k[3] for k in KITCHENS})} estados")

# 1. PERFIS DE ACESSO (tb_profile)

Existem apenas 3 perfis: `supervisor`, `estoquista` e `comprador`. Cada usuário recebe exatamente um deles.
O código fica em minúsculas porque é assim que a API grava e procura o perfil (`AccessType.toProfileCode`).

In [ ]:
# Só existem 3 perfis de acesso, e cada usuário recebe exatamente um deles.
# O código fica em minúsculas porque é assim que a API grava e procura o perfil
# (AccessType.toProfileCode). "SUPERVISOR" em maiúsculas duplicaria o papel.
PROFILES = [
    ("supervisor", "Supervisor — supervisiona compradores e estoquistas"),
    ("estoquista", "Estoquista — responsável por controlar o estoque"),
    ("comprador", "Comprador — responsável por comprar os produtos em falta no estoque"),
]

result = execute_values(
    cursor,
    """INSERT INTO tb_profile (access_type, description) VALUES %s
       ON CONFLICT (access_type) DO UPDATE SET description = EXCLUDED.description
       RETURNING id_profile, access_type""",
    PROFILES,
    fetch=True,
)
profile_id_by_code = {code: id_profile for id_profile, code in result}

# Trava: a tabela só pode conter os 3 perfis base.
cursor.execute("SELECT access_type FROM tb_profile ORDER BY access_type")
found = [r[0] for r in cursor.fetchall()]
expected = sorted(code for code, _ in PROFILES)
if found != expected:
    conn.rollback()
    raise RuntimeError(f"tb_profile deve ter só {expected}, mas tem: {found}")

conn.commit()
print(f"{len(profile_id_by_code)} perfis de acesso criados: {', '.join(code for code, _ in PROFILES)}")

# 2. COZINHAS INDUSTRIAIS (tb_kitchen)

O código da cozinha segue o gerador da API (`KitchenCodeGenerator`): 6 caracteres de um alfabeto sem O/0 e I/1.

In [ ]:
CODE_ALPHABET = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"

rows = []
kitchens = []
used_codes = set()
for company, unit, city, state, lat, lon, address, meals in KITCHENS:
    code = "".join(random.choices(CODE_ALPHABET, k=6))
    while code in used_codes:
        code = "".join(random.choices(CODE_ALPHABET, k=6))
    used_codes.add(code)

    created_at = work_datetime(date(2022, 1, 10) + timedelta(days=random.randint(0, 720)), 8, 18)
    accepting_requests = random.random() < 0.9
    rows.append((
        f"Cozinha Industrial {unit}", code, address, True, created_at,
        city, state, round(lat + random.uniform(-0.008, 0.008), 6), round(lon + random.uniform(-0.008, 0.008), 6),
        accepting_requests,
    ))
    kitchens.append({
        "company": company, "unit": unit, "city": city, "state": state, "meals": meals,
        "domain": COMPANIES[company]["domain"], "created_at": created_at,
    })

result = execute_values(
    cursor,
    """INSERT INTO tb_kitchen
        (name, code, address, active, created_at, city, state, latitude, longitude, accepting_requests)
       VALUES %s RETURNING id_kitchen""",
    rows,
    fetch=True,
)
for kitchen, (id_kitchen,) in zip(kitchens, result):
    kitchen["id"] = id_kitchen

conn.commit()
print(f"{len(kitchens)} cozinhas industriais criadas")

# 3. CATEGORIAS DE PRODUTO (tb_category, em árvore)

As 7 categorias oficiais são as **raízes** da árvore (`id_parent_category` nulo, V019). Os grupos de alimento entram um nível abaixo na seção 6.1, e cada produto aponta para o seu grupo. Grupos de produto sem categoria oficial correspondente (temperos, óleos, panificação, limpeza e descartáveis) ficam sem categoria (`id_category` é opcional).

In [ ]:
CATEGORIES = [
    ("Categoria Vegetal", "Vegetais, legumes e verduras."),
    ("Categoria Carne", "Carnes bovinas, suínas, aves e pescados."),
    ("Categoria Laticínio", "Leite, queijos e demais derivados do leite."),
    ("Categoria Grão", "Arroz, feijão, cereais e demais grãos."),
    ("Categoria Bebida", "Bebidas em geral."),
    ("Categoria Fruta", "Frutas in natura."),
    ("Categoria Doce", "Doces e sobremesas."),
]

result = execute_values(
    cursor,
    "INSERT INTO tb_category (name, description) VALUES %s RETURNING id_category, name",
    CATEGORIES,
    fetch=True,
)
category_id_by_name = {name: id_category for id_category, name in result}

conn.commit()
print(f"{len(category_id_by_name)} categorias de produto criadas")

# 4. UNIDADES DE MEDIDA (tb_measurement_unit)

As unidades em que o estoque é controlado no recebimento: peso e volume para itens fracionados e embalagens (caixa, fardo, galão...) para itens contados.

In [ ]:
UNITS = [
    ("KG", "Quilograma"), ("G", "Grama"), ("L", "Litro"), ("ML", "Mililitro"),
    ("UN", "Unidade"), ("DZ", "Dúzia"), ("PC", "Peça"), ("MC", "Maço"),
    ("CX", "Caixa"), ("PCT", "Pacote"), ("FD", "Fardo"), ("SC", "Saco"),
    ("BD", "Balde"), ("GL", "Galão"), ("BB", "Bombona"), ("LT", "Lata"),
    ("GF", "Garrafa"), ("FR", "Frasco"), ("PT", "Pote"), ("BIS", "Bisnaga"),
    ("BDJ", "Bandeja"), ("RL", "Rolo"),
]

result = execute_values(
    cursor,
    "INSERT INTO tb_measurement_unit (symbol, description) VALUES %s RETURNING id_unit, symbol",
    UNITS,
    fetch=True,
)
unit_id_by_symbol = {symbol: id_unit for id_unit, symbol in result}

conn.commit()
print(f"{len(unit_id_by_symbol)} unidades de medida criadas")

# 5. FORNECEDORES (tb_supplier)

As três divisões de alimentos do grupo que abastecem as cozinhas (Friboi, Seara e Swift) mais distribuidores regionais, cada um com seu segmento (frigorífico, hortifrúti, laticínios, embalagens...). O segmento define quais produtos o fornecedor pode fornecer na seção 8.

In [ ]:
INTERNAL_SUPPLIERS = [
    {"code": "FRIBOI", "legal_name": "JBS S.A. — Divisão Friboi (Fornecimento Interno)", "domain": "friboi.com.br",
     "prefix": "FRB", "brands": ["Friboi", "Maturatta"]},
    {"code": "SEARA", "legal_name": "Seara Alimentos Ltda. (Fornecimento Interno)", "domain": "seara.com.br",
     "prefix": "SEA", "brands": ["Seara", "Seara Gourmet"]},
    {"code": "SWIFT", "legal_name": "JBS S.A. — Divisão Swift (Fornecimento Interno)", "domain": "swift.com.br",
     "prefix": "SWF", "brands": ["Swift"]},
]

# segmento -> grupos de produto que fornece, prazo de entrega (dias), sufixo do domínio e modelos de razão social
SUPPLIER_SEGMENTS = {
    "frigorifico": {"groups": ["beef", "pork"], "lead": (2, 5), "domain": "carnes", "weight": 7, "templates": [
        "Frigorífico {n} Ltda.", "{n} Carnes e Derivados Ltda.", "{n} Distribuidora de Carnes Ltda."]},
    "avicola": {"groups": ["poultry"], "lead": (2, 5), "domain": "avicola", "weight": 5, "templates": [
        "Avícola {n} Ltda.", "{n} Alimentos Avícolas Ltda.", "Granja {n} Comércio de Aves Ltda."]},
    "pescados": {"groups": ["fish"], "lead": (3, 7), "domain": "pescados", "weight": 4, "templates": [
        "{n} Pescados Ltda.", "Pescados {n} Comércio e Importação Ltda.", "{n} Frutos do Mar Ltda."]},
    "laticinios": {"groups": ["dairy"], "lead": (2, 6), "domain": "laticinios", "weight": 8, "templates": [
        "Laticínios {n} Ltda.", "{n} Frios e Laticínios Ltda.", "Distribuidora de Laticínios {n} Ltda."]},
    "hortifruti": {"groups": ["produce_veg", "produce_fruit"], "lead": (1, 2), "domain": "hortifruti", "weight": 14, "templates": [
        "Hortifrúti {n} Ltda.", "{n} Comércio de Frutas e Verduras Ltda.", "Central de Abastecimento {n} Ltda.",
        "Cooperativa Agrícola {n}"]},
    "cerealista": {"groups": ["grains"], "lead": (3, 8), "domain": "cereais", "weight": 7, "templates": [
        "Cerealista {n} Ltda.", "{n} Grãos e Cereais Ltda.", "Beneficiadora de Arroz {n} Ltda.", "Moinho {n} S.A."]},
    "atacado": {"groups": ["grains", "seasoning", "oils", "sweets", "beverages"], "lead": (3, 7), "domain": "alimentos",
                "weight": 14, "templates": [
        "Atacadista {n} Ltda.", "{n} Distribuidora de Alimentos Ltda.", "Comercial {n} de Alimentos Ltda.",
        "{n} Food Service Ltda."]},
    "bebidas": {"groups": ["beverages"], "lead": (2, 6), "domain": "bebidas", "weight": 6, "templates": [
        "{n} Distribuidora de Bebidas Ltda.", "Bebidas {n} Ltda.", "Águas e Bebidas {n} Ltda."]},
    "doces": {"groups": ["sweets"], "lead": (4, 10), "domain": "doces", "weight": 4, "templates": [
        "Doces {n} Ltda.", "{n} Indústria de Doces e Conservas Ltda.", "Confeitaria Industrial {n} Ltda."]},
    "panificacao": {"groups": ["bakery"], "lead": (1, 2), "domain": "paes", "weight": 5, "templates": [
        "Panificadora {n} Ltda.", "{n} Pães e Massas Ltda.", "Indústria de Panificação {n} Ltda."]},
    "condimentos": {"groups": ["seasoning", "oils"], "lead": (4, 10), "domain": "temperos", "weight": 5, "templates": [
        "{n} Temperos e Condimentos Ltda.", "Empório {n} Especiarias Ltda."]},
    "limpeza": {"groups": ["cleaning"], "lead": (5, 12), "domain": "limpeza", "weight": 9, "templates": [
        "{n} Produtos de Limpeza Ltda.", "{n} Química e Higienização Ltda.", "Distribuidora {n} de Higiene e Limpeza Ltda."]},
    "embalagens": {"groups": ["disposables"], "lead": (5, 15), "domain": "embalagens", "weight": 8, "templates": [
        "{n} Embalagens Ltda.", "{n} Descartáveis Ltda.", "Plásticos {n} Indústria e Comércio Ltda."]},
}

PLACE_ROOTS = [
    "Vale do Araguaia", "Serra Azul", "Santa Helena", "Bom Jesus", "Três Fronteiras", "Rio Pardo", "Nova Aliança",
    "Boa Vista", "Planalto Central", "Pantanal", "Cerrado Mineiro", "Alto Paraná", "Campos Gerais", "Mantiqueira",
    "Vale do Itajaí", "Oeste Paulista", "Litoral Sul", "Serra Gaúcha", "Chapada", "Paranapanema", "Rio Doce",
    "Guaporé", "Xingu", "Iguaçu", "Pampa", "Capricórnio", "Progresso", "Estrela do Sul", "Primavera", "Horizonte",
    "Bandeirantes", "Guarani", "Ipê Amarelo", "Jequitibá", "Aroeira", "Figueira", "São Jorge", "São Bento",
    "Santa Rita", "São Judas Tadeu", "Santo Antônio", "São Mateus", "São Lucas", "São Cristóvão", "Santa Fé",
    "São Miguel", "São Gabriel", "Santa Cecília", "São José", "Nova Esperança", "Vale Verde", "Monte Alegre",
    "Água Limpa", "Campo Belo", "Terra Roxa", "Sol Nascente", "Dois Irmãos", "Bela Vista", "Ouro Verde",
    "Mato Grosso", "Rio Claro", "Lagoa Dourada", "Morro Alto", "Canaã", "Pioneira", "Fronteira Oeste",
]
SURNAMES = [
    "Bertolini", "Zanella", "Moretti", "Albuquerque", "Carvalho", "Fontana", "Rossato", "Pasqualotto", "Schneider",
    "Kuhn", "Tonin", "Dalla Costa", "Takahashi", "Nakamura", "Bortolozzo", "Venturini", "Marchetti", "Ferraz",
    "Andrade", "Prado", "Barreto", "Monteiro", "Siqueira", "Toledo", "Queiroz", "Valadares", "Bianchi", "Lorenzetti",
    "Hoffmann", "Weber", "Zimmermann", "Battistella", "Menegatti", "Sakamoto", "Yamamoto", "Furlan", "Gasparini",
    "Brandão", "Cavalcanti", "Figueiredo", "Guimarães", "Peixoto", "Sampaio", "Teixeira", "Vasconcelos",
]


def supplier_root():
    roll = random.random()
    if roll < 0.62:
        return random.choice(PLACE_ROOTS)
    if roll < 0.80:
        return random.choice(SURNAMES)
    if roll < 0.92:
        return f"Irmãos {random.choice(SURNAMES)}"
    first, second = random.sample(SURNAMES, 2)
    return f"{first} & {second}"


rows = []
suppliers = []
used_cnpj = set()
used_names = set()


def new_cnpj():
    cnpj = fake.cnpj()
    while cnpj in used_cnpj:
        cnpj = fake.cnpj()
    used_cnpj.add(cnpj)
    return cnpj


for internal in INTERNAL_SUPPLIERS:
    rows.append((
        internal["legal_name"], new_cnpj(), f"pedidos.internos@{internal['domain']}",
        f"+55 11 9{random.randint(7000, 9999)}-{random.randint(1000, 9999)}", 5, True,
        work_datetime(date(2021, 3, 1) + timedelta(days=random.randint(0, 120))),
    ))
    suppliers.append({"segment": "internal", "groups": [], "brands": internal["brands"], "lead": (2, 4),
                      "prefix": internal["prefix"], "active": True})

segment_keys = list(SUPPLIER_SEGMENTS)
segment_weights = [SUPPLIER_SEGMENTS[k]["weight"] for k in segment_keys]
while len(suppliers) < ROWS_PER_TABLE:
    segment = random.choices(segment_keys, weights=segment_weights)[0]
    spec = SUPPLIER_SEGMENTS[segment]
    root = supplier_root()
    legal_name = random.choice(spec["templates"]).format(n=root)
    if legal_name in used_names:
        continue
    used_names.add(legal_name)

    state = random.choice(KITCHENS)[3]
    domain = f"{slugify(root.replace('&', 'e'), '')}{spec['domain']}.com.br"
    email = f"{random.choice(['comercial', 'vendas', 'contato', 'pedidos', 'atendimento'])}@{domain}"
    whatsapp = f"+55 {random.choice(DDD_BY_UF[state])} 9{random.randint(7000, 9999)}-{random.randint(1000, 9999)}"
    rating = random.choices([1, 2, 3, 4, 5], weights=[2, 5, 18, 45, 30])[0]
    active = random.random() < 0.95
    created_at = work_datetime(date(2020, 6, 1) + timedelta(days=random.randint(0, 1900)))
    # prefixo do código do produto no fornecedor: começo do nome que identifica a empresa (ex.: NAK-48213)
    skip = {"e", "do", "da", "de", "dos", "das", "irmaos", "sao", "santa", "santo"}
    prefix = [w for w in slugify(root, " ").split() if w not in skip][0][:3].upper()

    rows.append((legal_name, new_cnpj(), email, whatsapp, rating, active, created_at))
    suppliers.append({"segment": segment, "groups": spec["groups"], "brands": [], "lead": spec["lead"],
                      "prefix": prefix, "active": active})

result = execute_values(
    cursor,
    """INSERT INTO tb_supplier (legal_name, cnpj, email, whatsapp, rating, active, created_at)
       VALUES %s RETURNING id_supplier""",
    rows,
    fetch=True,
)
for supplier, (id_supplier,) in zip(suppliers, result):
    supplier["id"] = id_supplier

conn.commit()
print(f"{len(suppliers)} fornecedores criados ({len(INTERNAL_SUPPLIERS)} divisões internas do grupo J&F)")

# 6. PRODUTOS (tb_product)

Catálogo de referência com os itens de uma cozinha industrial: para cada item, as marcas, as embalagens de compra, o preço de referência, o consumo médio e a validade. Ele é a base das duas fontes de produto: os produtos reais da Open Food Facts (seção 6.1) e as combinações curadas item + marca + embalagem (seção 6.2). Hortifrúti a granel não tem código de barras, como na prática.

In [ ]:
def item(name, price, use, brands, packs, life=None):
    """price: R$ por unidade-base (kg, litro ou unidade); use: consumo diário por 1.000 refeições, na
    unidade-base; packs: (rótulo, unidade de estoque, conteúdo da unidade de estoque em unidade-base);
    life: validade em dias (mín, máx), ou None para itens sem validade."""
    return {"name": name, "price": price, "use": use, "brands": brands, "packs": packs, "life": life}


CHILLED = [("Peça a vácuo", "KG", 1), ("Caixa 20 kg", "KG", 1)]
FROZEN_BOX = [("Caixa 10 kg", "KG", 1), ("Caixa 20 kg", "KG", 1)]
FROZEN_PACK = [("Pacote 1 kg", "KG", 1), ("Caixa 10 kg", "KG", 1)]
PORK_FROZEN = [("Caixa 10 kg", "KG", 1), ("Pacote 5 kg", "KG", 1)]
PORK_VACUUM = [("Peça a vácuo", "KG", 1), ("Pacote 2,5 kg", "KG", 1)]
POULTRY = [("Caixa 18 kg", "KG", 1), ("Caixa 12 kg", "KG", 1), ("Pacote 1 kg", "KG", 1)]
FISH = [("Caixa 10 kg", "KG", 1), ("Pacote 1 kg", "KG", 1)]
CHEESE = [("Peça 4 kg", "KG", 1), ("Fatiado — Pacote 1 kg", "KG", 1)]
RICE = [("Fardo 30 kg (6 × 5 kg)", "FD", 30), ("Saco 25 kg", "SC", 25)]
BEANS = [("Fardo 30 kg (30 × 1 kg)", "FD", 30), ("Saco 30 kg", "SC", 30)]
PASTA = [("Caixa 10 kg (20 × 500 g)", "CX", 10), ("Pacote 5 kg", "PCT", 5)]
SODA = [("Fardo 6 × 2 L", "FD", 12), ("Fardo 12 latas de 350 ml", "FD", 4.2)]
COFFEE = [("Fardo 20 × 500 g", "FD", 10), ("Pacote 500 g", "PCT", 0.5)]
BOX_10KG, BOX_15KG, BOX_20KG = [("Caixa 10 kg", "KG", 1)], [("Caixa 15 kg", "KG", 1)], [("Caixa 20 kg", "KG", 1)]
BUNCH, HEAD = [("Maço", "MC", 1)], [("Unidade (pé)", "UN", 1)]
GALLON_5L = [("Galão 5 L", "GL", 5)]

CHILLED_LIFE, FROZEN_LIFE = (35, 50), (300, 540)

# group -> rótulo, categoria oficial (ou None), quantos produtos entram no catálogo e itens
FOOD_GROUPS = {
    "beef": {"label": "Carnes Bovinas", "category": "Categoria Carne", "target": 56, "items": [
        item("Picanha Bovina Resfriada", 69.90, 3, ["Friboi", "Maturatta", "Swift"], CHILLED, CHILLED_LIFE),
        item("Filé Mignon Bovino Resfriado", 82.90, 2, ["Friboi", "Maturatta"], CHILLED, CHILLED_LIFE),
        item("Contrafilé Bovino Resfriado", 45.90, 8, ["Friboi", "Maturatta", "Swift"], CHILLED, CHILLED_LIFE),
        item("Alcatra Bovina Resfriada", 46.90, 8, ["Friboi", "Maturatta", "Swift"], CHILLED, CHILLED_LIFE),
        item("Maminha Bovina Resfriada", 43.90, 4, ["Friboi", "Maturatta"], CHILLED, CHILLED_LIFE),
        item("Fraldinha Bovina Resfriada", 41.90, 5, ["Friboi", "Maturatta", "Swift"], CHILLED, CHILLED_LIFE),
        item("Patinho Bovino Resfriado", 38.90, 14, ["Friboi", "Swift"], CHILLED, CHILLED_LIFE),
        item("Coxão Mole Bovino Resfriado", 39.50, 14, ["Friboi", "Swift"], CHILLED, CHILLED_LIFE),
        item("Coxão Duro Bovino Resfriado", 36.90, 10, ["Friboi"], CHILLED, CHILLED_LIFE),
        item("Lagarto Bovino Resfriado", 38.50, 7, ["Friboi", "Swift"], CHILLED, CHILLED_LIFE),
        item("Acém Bovino Resfriado", 29.90, 18, ["Friboi", "Swift"], CHILLED, CHILLED_LIFE),
        item("Cupim Bovino Resfriado", 35.90, 3, ["Friboi", "Maturatta"], CHILLED, CHILLED_LIFE),
        item("Músculo Bovino Congelado", 28.90, 10, ["Friboi", "Swift"], FROZEN_BOX, FROZEN_LIFE),
        item("Paleta Bovina Congelada", 30.90, 9, ["Friboi"], FROZEN_BOX, FROZEN_LIFE),
        item("Costela Bovina Congelada", 24.90, 7, ["Friboi", "Swift"], FROZEN_BOX, FROZEN_LIFE),
        item("Carne Moída Bovina Congelada", 27.90, 16, ["Friboi", "Swift"], FROZEN_PACK, FROZEN_LIFE),
        item("Carne Bovina em Cubos Congelada", 33.90, 10, ["Friboi", "Swift"], FROZEN_PACK, FROZEN_LIFE),
        item("Bife de Contrafilé Porcionado Congelado", 51.90, 5, ["Friboi", "Swift"], FROZEN_PACK, FROZEN_LIFE),
        item("Fígado Bovino Congelado", 17.90, 3, ["Friboi"], FROZEN_BOX, FROZEN_LIFE),
    ]},
    "pork": {"label": "Carnes Suínas e Frios", "category": "Categoria Carne", "target": 32, "items": [
        item("Pernil Suíno sem Osso Congelado", 21.90, 9, ["Seara", "Swift"], PORK_FROZEN, FROZEN_LIFE),
        item("Lombo Suíno Resfriado", 26.90, 7, ["Seara", "Swift"], PORK_VACUUM, (25, 40)),
        item("Bisteca Suína Congelada", 19.90, 7, ["Seara"], PORK_FROZEN, FROZEN_LIFE),
        item("Costela Suína Congelada", 23.90, 5, ["Seara", "Swift"], PORK_FROZEN, FROZEN_LIFE),
        item("Paleta Suína sem Osso Congelada", 18.90, 7, ["Seara"], PORK_FROZEN, FROZEN_LIFE),
        item("Bacon em Manta Defumado", 41.90, 2, ["Seara", "Swift"], PORK_VACUUM, (60, 90)),
        item("Linguiça Toscana Congelada", 18.90, 10, ["Seara", "Swift"], PORK_FROZEN, (180, 240)),
        item("Linguiça Calabresa Defumada", 26.90, 5, ["Seara", "Swift"], PORK_VACUUM, (60, 90)),
        item("Filé Mignon Suíno Congelado", 31.90, 3, ["Seara"], PORK_FROZEN, FROZEN_LIFE),
        item("Presunto Cozido sem Capa de Gordura", 32.90, 4, ["Seara"], [("Peça 3,5 kg", "KG", 1)], (60, 90)),
        item("Mortadela Defumada", 19.90, 3, ["Seara"], [("Peça 5 kg", "KG", 1)], (60, 90)),
        item("Salame Tipo Italiano", 79.90, 0.5, ["Seara"], [("Peça a vácuo", "KG", 1)], (90, 150)),
        item("Copa Lombo Suína Resfriada", 25.90, 4, ["Seara", "Swift"], PORK_VACUUM, (25, 40)),
    ]},
    "poultry": {"label": "Aves", "category": "Categoria Carne", "target": 44, "items": [
        item("Peito de Frango sem Osso Congelado", 18.90, 28, ["Seara"], POULTRY, (300, 450)),
        item("Filé de Peito de Frango IQF", 21.90, 22, ["Seara"], POULTRY, (300, 450)),
        item("Coxa e Sobrecoxa de Frango Congelada", 12.90, 22, ["Seara"], POULTRY, (300, 450)),
        item("Sobrecoxa de Frango sem Osso Congelada", 17.90, 18, ["Seara"], POULTRY, (300, 450)),
        item("Coxa de Frango Congelada", 11.90, 14, ["Seara"], POULTRY, (300, 450)),
        item("Asa de Frango Congelada", 14.90, 5, ["Seara"], POULTRY, (300, 450)),
        item("Coxinha da Asa Congelada", 17.90, 5, ["Seara"], POULTRY, (300, 450)),
        item("Filé de Sassami Congelado", 22.90, 5, ["Seara"], POULTRY, (300, 450)),
        item("Frango em Cubos Congelado", 20.90, 10, ["Seara"], POULTRY, (300, 450)),
        item("Frango a Passarinho Congelado", 15.90, 6, ["Seara"], POULTRY, (300, 450)),
        item("Frango Inteiro Congelado", 10.90, 8, ["Seara"], [("Caixa 18 kg", "KG", 1), ("Unidade ~2,2 kg", "KG", 1)], (300, 450)),
        item("Coração de Frango Congelado", 26.90, 2, ["Seara"], [("Pacote 1 kg", "KG", 1), ("Caixa 12 kg", "KG", 1)], (300, 450)),
        item("Moela de Frango Congelada", 12.90, 2, ["Seara"], [("Pacote 1 kg", "KG", 1), ("Caixa 12 kg", "KG", 1)], (300, 450)),
        item("Peito de Peru Defumado", 54.90, 1, ["Seara"], [("Peça 2,5 kg", "KG", 1)], (60, 90)),
        item("Steak de Frango Empanado", 25.90, 6, ["Seara", "Seara Gourmet"], [("Caixa 10 kg", "KG", 1), ("Pacote 1 kg", "KG", 1)], (240, 365)),
        item("Filé de Frango Empanado", 24.90, 5, ["Seara", "Seara Gourmet"], [("Caixa 10 kg", "KG", 1), ("Pacote 1 kg", "KG", 1)], (240, 365)),
        item("Nuggets de Frango Tradicional", 23.90, 5, ["Seara"], [("Caixa 10 kg", "KG", 1), ("Pacote 1 kg", "KG", 1)], (240, 365)),
    ]},
    "fish": {"label": "Peixes e Frutos do Mar", "category": "Categoria Carne", "target": 24, "items": [
        item("Filé de Tilápia Congelado", 42.90, 8, ["Frescatto", "Costa Sul"], FISH, (360, 540)),
        item("Filé de Merluza Congelado", 34.90, 8, ["Frescatto", "Costa Sul"], FISH, (360, 540)),
        item("Filé de Pescada Branca Congelado", 36.90, 5, ["Frescatto", "Costa Sul"], FISH, (360, 540)),
        item("Posta de Cação Congelada", 38.90, 3, ["Costa Sul"], FISH, (360, 540)),
        item("Filé de Salmão Congelado", 94.90, 1.5, ["Frescatto"], FISH, (360, 540)),
        item("Camarão Cinza Descascado Congelado", 72.90, 1.5, ["Frescatto", "Costa Sul"], FISH, (360, 540)),
        item("Lombo de Bacalhau Dessalgado Congelado", 109.90, 1, ["Frescatto"], [("Caixa 5 kg", "KG", 1), ("Pacote 1 kg", "KG", 1)], (360, 540)),
        item("Isca de Peixe Empanada", 31.90, 3, ["Frescatto", "Costa Sul"], FISH, (300, 450)),
        item("Sardinha em Óleo", 32.00, 1, ["Gomes da Costa", "Coqueiro"], [("Caixa 24 latas de 125 g", "CX", 3)], (600, 900)),
        item("Atum Sólido em Óleo", 64.00, 0.8, ["Gomes da Costa", "Coqueiro"], [("Lata 1,7 kg", "LT", 1.7)], (600, 900)),
    ]},
    "dairy": {"label": "Laticínios", "category": "Categoria Laticínio", "target": 56, "items": [
        item("Leite Integral UHT", 4.89, 35, ["Piracanjuba", "Itambé", "Vigor", "Parmalat"], [("Caixa 12 × 1 L", "CX", 12)], (120, 180)),
        item("Leite Desnatado UHT", 5.09, 8, ["Piracanjuba", "Itambé", "Vigor"], [("Caixa 12 × 1 L", "CX", 12)], (120, 180)),
        item("Leite em Pó Integral", 38.90, 1.5, ["Itambé", "Piracanjuba", "Ninho"], [("Pacote 1 kg", "PCT", 1)], (300, 540)),
        item("Queijo Muçarela", 41.90, 6, ["Tirolez", "Président", "Scala", "Quatá"], CHEESE, (45, 75)),
        item("Queijo Prato", 44.90, 4, ["Tirolez", "Quatá", "Scala"], CHEESE, (45, 75)),
        item("Queijo Parmesão Ralado", 72.90, 0.8, ["Tirolez", "Président", "Scala"], [("Pacote 1 kg", "PCT", 1)], (90, 150)),
        item("Queijo Minas Frescal", 36.90, 2, ["Tirolez", "Quatá", "Polenghi"], [("Peça 500 g", "PC", 0.5)], (20, 30)),
        item("Queijo Coalho em Barra", 49.90, 1, ["Tirolez", "Quatá"], [("Barra 1 kg", "KG", 1)], (40, 60)),
        item("Requeijão Cremoso", 31.90, 3, ["Catupiry", "Vigor", "Itambé", "Poços de Caldas"], [("Bisnaga 1,5 kg", "BIS", 1.5), ("Balde 3,6 kg", "BD", 3.6)], (60, 90)),
        item("Manteiga com Sal", 56.90, 1.5, ["Aviação", "Président", "Itambé", "Vigor"], [("Pote 500 g", "PT", 0.5), ("Bloco 5 kg", "KG", 1)], (90, 150)),
        item("Margarina com Sal 80% Lipídios", 15.90, 4, ["Qualy", "Delícia"], [("Balde 15 kg", "BD", 15), ("Pote 1 kg", "PT", 1)], (120, 180)),
        item("Creme de Leite UHT", 18.90, 3, ["Piracanjuba", "Itambé", "Nestlé"], [("Caixa 27 × 200 g", "CX", 5.4), ("Embalagem 1,01 kg", "UN", 1.01)], (180, 270)),
        item("Iogurte Natural Integral", 13.90, 5, ["Vigor", "Itambé", "Batavo"], [("Balde 5 kg", "BD", 5)], (25, 40)),
        item("Ricota Fresca", 31.90, 1, ["Tirolez", "Quatá"], [("Peça 500 g", "PC", 0.5)], (15, 25)),
        item("Cream Cheese", 42.90, 1, ["Polenghi", "Philadelphia"], [("Pote 1,2 kg", "PT", 1.2)], (60, 90)),
    ]},
    "produce_veg": {"label": "Legumes e Verduras", "category": "Categoria Vegetal", "target": 48, "items": [
        item("Tomate Italiano", 7.90, 22, [None], BOX_20KG, (7, 12)),
        item("Tomate Salada Longa Vida", 6.90, 18, [None], BOX_20KG, (8, 14)),
        item("Tomate Cereja", 19.90, 2, [None], [("Bandeja 500 g", "BDJ", 0.5)], (6, 10)),
        item("Cebola Amarela", 4.90, 18, [None], [("Saco 20 kg", "KG", 1)], (20, 40)),
        item("Cebola Roxa", 6.50, 4, [None], [("Saco 10 kg", "KG", 1)], (20, 40)),
        item("Batata Inglesa Lavada", 5.40, 30, [None], [("Saco 25 kg", "KG", 1)], (15, 30)),
        item("Batata Bolinha", 6.90, 6, [None], [("Saco 10 kg", "KG", 1)], (15, 30)),
        item("Batata-Doce Rosada", 4.80, 10, [None], BOX_20KG, (15, 30)),
        item("Cenoura Extra AA", 4.60, 14, [None], BOX_20KG, (12, 25)),
        item("Beterraba Extra", 4.90, 8, [None], BOX_20KG, (12, 25)),
        item("Abobrinha Italiana", 5.90, 8, [None], BOX_15KG, (7, 12)),
        item("Abóbora Cabotiá", 4.20, 8, [None], [("Saco 20 kg", "KG", 1)], (20, 45)),
        item("Chuchu", 3.90, 8, [None], BOX_20KG, (8, 14)),
        item("Pimentão Verde", 8.90, 5, [None], BOX_10KG, (7, 12)),
        item("Pimentão Vermelho", 14.90, 3, [None], BOX_10KG, (7, 12)),
        item("Pimentão Amarelo", 15.90, 2, [None], BOX_10KG, (7, 12)),
        item("Pepino Japonês", 6.90, 5, [None], BOX_15KG, (7, 12)),
        item("Berinjela", 6.40, 4, [None], BOX_15KG, (7, 12)),
        item("Quiabo", 11.90, 3, [None], [("Caixa 12 kg", "KG", 1)], (4, 7)),
        item("Vagem Macarrão", 12.90, 4, [None], BOX_10KG, (5, 8)),
        item("Jiló", 9.90, 2, [None], BOX_10KG, (5, 8)),
        item("Brócolis Ninja", 14.90, 6, [None], BOX_10KG, (5, 8)),
        item("Couve-Flor", 9.90, 5, [None], BOX_10KG, (5, 8)),
        item("Repolho Verde", 3.20, 12, [None], [("Saco 20 kg", "KG", 1)], (10, 20)),
        item("Repolho Roxo", 4.90, 4, [None], [("Saco 20 kg", "KG", 1)], (10, 20)),
        item("Alface Crespa", 2.90, 60, [None], HEAD, (3, 6)),
        item("Alface Americana", 4.50, 30, [None], HEAD, (4, 7)),
        item("Rúcula", 3.50, 20, [None], BUNCH, (3, 5)),
        item("Agrião", 3.50, 12, [None], BUNCH, (2, 4)),
        item("Couve-Manteiga", 3.20, 20, [None], BUNCH, (3, 5)),
        item("Espinafre", 4.20, 10, [None], BUNCH, (3, 5)),
        item("Cheiro-Verde", 2.50, 25, [None], BUNCH, (3, 5)),
        item("Coentro", 2.80, 10, [None], BUNCH, (3, 5)),
        item("Cebolinha", 3.00, 10, [None], BUNCH, (3, 5)),
        item("Hortelã", 3.50, 4, [None], BUNCH, (3, 5)),
        item("Salsão", 6.90, 2, [None], BUNCH, (5, 8)),
        item("Alho-Poró", 6.90, 3, [None], BUNCH, (7, 12)),
        item("Rabanete", 4.90, 2, [None], BUNCH, (5, 8)),
        item("Alho Nacional Roxo", 29.90, 3, [None], BOX_10KG, (60, 120)),
        item("Alho Descascado Resfriado", 34.90, 2, [None], [("Pacote 1 kg", "PCT", 1)], (10, 20)),
        item("Gengibre", 18.90, 1, [None], BOX_10KG, (20, 40)),
        item("Mandioquinha", 12.90, 4, [None], BOX_10KG, (7, 12)),
        item("Inhame", 7.90, 4, [None], BOX_20KG, (20, 40)),
        item("Milho Verde em Espiga", 1.60, 20, [None], [("Saco com 50 espigas", "SC", 50)], (4, 7)),
        item("Mandioca Descascada Congelada", 8.90, 10, [None], [("Pacote 2 kg", "PCT", 2)], (180, 365)),
        item("Ervilha Congelada", 14.90, 4, [None], [("Pacote 2 kg", "PCT", 2)], (240, 365)),
        item("Seleta de Legumes Congelada", 12.90, 6, [None], [("Pacote 2 kg", "PCT", 2)], (240, 365)),
        item("Couve Picada Higienizada", 19.90, 4, [None], [("Pacote 1 kg", "PCT", 1)], (4, 6)),
    ]},
    "produce_fruit": {"label": "Frutas", "category": "Categoria Fruta", "target": 37, "items": [
        item("Banana Prata", 5.90, 25, [None], BOX_20KG, (5, 9)),
        item("Banana Nanica", 4.50, 15, [None], BOX_20KG, (5, 9)),
        item("Maçã Gala", 9.90, 15, [None], [("Caixa 18 kg", "KG", 1)], (20, 40)),
        item("Maçã Fuji", 11.90, 8, [None], [("Caixa 18 kg", "KG", 1)], (20, 40)),
        item("Laranja Pera", 3.90, 25, [None], [("Saco 20 kg", "KG", 1)], (15, 25)),
        item("Laranja Lima", 5.50, 5, [None], [("Saco 20 kg", "KG", 1)], (15, 25)),
        item("Limão Tahiti", 5.90, 6, [None], BOX_20KG, (15, 25)),
        item("Mamão Formosa", 5.90, 15, [None], BOX_15KG, (5, 9)),
        item("Mamão Papaya", 8.90, 6, [None], BOX_10KG, (5, 9)),
        item("Melancia", 2.90, 30, [None], [("A granel", "KG", 1)], (10, 18)),
        item("Melão Amarelo", 6.90, 10, [None], [("Caixa 13 kg", "KG", 1)], (12, 20)),
        item("Melão Cantaloupe", 9.90, 3, [None], [("Caixa 10 kg", "KG", 1)], (8, 14)),
        item("Abacaxi Pérola", 6.90, 10, [None], [("Unidade", "UN", 1)], (5, 9)),
        item("Manga Palmer", 6.90, 10, [None], [("Caixa 6 kg", "KG", 1)], (6, 10)),
        item("Manga Tommy", 5.90, 6, [None], [("Caixa 6 kg", "KG", 1)], (6, 10)),
        item("Uva Thompson sem Semente", 16.90, 3, [None], [("Caixa 8 kg", "KG", 1)], (7, 12)),
        item("Uva Niágara", 12.90, 3, [None], [("Caixa 6 kg", "KG", 1)], (4, 7)),
        item("Morango", 24.90, 2, [None], [("Bandeja 250 g", "BDJ", 0.25)], (3, 5)),
        item("Abacate", 8.90, 3, [None], BOX_20KG, (6, 10)),
        item("Pera Williams", 14.90, 3, [None], [("Caixa 18 kg", "KG", 1)], (15, 25)),
        item("Tangerina Ponkan", 5.90, 10, [None], BOX_20KG, (10, 18)),
        item("Goiaba Vermelha", 8.90, 4, [None], BOX_10KG, (5, 8)),
        item("Maracujá Azedo", 9.90, 3, [None], [("Caixa 12 kg", "KG", 1)], (10, 18)),
        item("Kiwi", 24.90, 1, [None], BOX_10KG, (15, 30)),
        item("Caqui Rama Forte", 11.90, 2, [None], [("Caixa 6 kg", "KG", 1)], (7, 12)),
        item("Ameixa Nacional", 19.90, 1, [None], [("Caixa 6 kg", "KG", 1)], (7, 12)),
        item("Pêssego", 19.90, 1, [None], [("Caixa 6 kg", "KG", 1)], (5, 8)),
        item("Coco Verde", 5.50, 3, [None], [("Unidade", "UN", 1)], (10, 20)),
        item("Polpa de Maracujá Congelada", 19.90, 3, ["Mais Fruta", "DeMarchi"], [("Pacote 1 kg", "PCT", 1)], (300, 365)),
        item("Polpa de Acerola Congelada", 14.90, 3, ["Mais Fruta", "DeMarchi"], [("Pacote 1 kg", "PCT", 1)], (300, 365)),
        item("Polpa de Goiaba Congelada", 14.90, 2, ["Mais Fruta", "DeMarchi"], [("Pacote 1 kg", "PCT", 1)], (300, 365)),
        item("Polpa de Manga Congelada", 15.90, 2, ["Mais Fruta", "DeMarchi"], [("Pacote 1 kg", "PCT", 1)], (300, 365)),
        item("Polpa de Caju Congelada", 13.90, 2, ["DeMarchi"], [("Pacote 1 kg", "PCT", 1)], (300, 365)),
    ]},
    "grains": {"label": "Grãos, Cereais e Massas", "category": "Categoria Grão", "target": 53, "items": [
        item("Arroz Branco Tipo 1", 6.20, 65, ["Camil", "Tio João", "Prato Fino"], RICE, (300, 420)),
        item("Arroz Parboilizado Tipo 1", 6.40, 25, ["Camil", "Tio João", "Namorado"], RICE, (300, 420)),
        item("Arroz Integral Tipo 1", 8.90, 4, ["Camil", "Tio João"], [("Fardo 10 × 1 kg", "FD", 10)], (240, 360)),
        item("Feijão Carioca Tipo 1", 8.40, 32, ["Camil", "Kicaldo", "Namorado"], BEANS, (180, 300)),
        item("Feijão Preto Tipo 1", 8.90, 12, ["Camil", "Kicaldo"], BEANS, (180, 300)),
        item("Lentilha", 14.90, 1, ["Camil", "Yoki"], [("Fardo 20 × 500 g", "FD", 10)], (300, 420)),
        item("Grão-de-Bico", 16.90, 1.5, ["Camil", "Yoki"], [("Fardo 20 × 500 g", "FD", 10)], (300, 420)),
        item("Ervilha Partida", 10.90, 1, ["Yoki"], [("Fardo 20 × 500 g", "FD", 10)], (300, 420)),
        item("Farinha de Trigo Tipo 1", 4.90, 10, ["Dona Benta", "Renata", "Sol"], [("Saco 25 kg", "SC", 25), ("Fardo 10 × 1 kg", "FD", 10)], (120, 180)),
        item("Farinha de Mandioca Torrada", 9.90, 3, ["Yoki", "Kicaldo"], [("Fardo 10 × 1 kg", "FD", 10)], (180, 300)),
        item("Farinha de Milho Amarela", 9.40, 1.5, ["Yoki"], [("Fardo 10 × 500 g", "FD", 5)], (180, 300)),
        item("Fubá Mimoso", 5.90, 3, ["Yoki", "Sinhá"], [("Fardo 10 × 1 kg", "FD", 10)], (180, 300)),
        item("Macarrão Espaguete Nº 8", 8.90, 12, ["Renata", "Adria", "Barilla"], PASTA, (360, 540)),
        item("Macarrão Parafuso", 8.90, 8, ["Renata", "Adria"], PASTA, (360, 540)),
        item("Macarrão Penne", 9.90, 6, ["Renata", "Barilla"], PASTA, (360, 540)),
        item("Massa para Lasanha", 14.90, 2, ["Renata", "Barilla"], [("Caixa 10 × 500 g", "CX", 5)], (360, 540)),
        item("Aveia em Flocos Finos", 16.90, 1, ["Quaker", "Yoki"], [("Pacote 1 kg", "PCT", 1)], (240, 360)),
        item("Farinha de Rosca", 11.90, 1.5, ["Yoki"], [("Pacote 1 kg", "PCT", 1)], (180, 300)),
        item("Canjica Branca", 9.90, 1, ["Yoki"], [("Fardo 10 × 500 g", "FD", 5)], (300, 420)),
        item("Amido de Milho", 13.90, 1, ["Maizena", "Yoki"], [("Caixa 1 kg", "CX", 1)], (360, 540)),
        item("Polvilho Doce", 12.90, 1, ["Yoki"], [("Pacote 1 kg", "PCT", 1)], (300, 420)),
    ]},
    "sweets": {"label": "Doces e Sobremesas", "category": "Categoria Doce", "target": 30, "items": [
        item("Doce de Leite Pastoso", 26.90, 1.5, ["Itambé", "Aviação", "Viçosa"], [("Balde 4,8 kg", "BD", 4.8), ("Pote 1 kg", "PT", 1)], (180, 270)),
        item("Goiabada Cascão", 19.90, 1.5, ["Predilecta", "Zélia"], [("Bloco 2,5 kg", "UN", 2.5)], (300, 365)),
        item("Leite Condensado", 21.90, 3, ["Moça", "Piracanjuba", "Itambé"], [("Caixa 27 × 395 g", "CX", 10.665), ("Bag 2,5 kg", "UN", 2.5)], (240, 300)),
        item("Gelatina em Pó Sabor Morango", 22.90, 2, ["Royal", "Dr. Oetker"], [("Pacote 1 kg", "PCT", 1)], (360, 540)),
        item("Gelatina em Pó Sabor Uva", 22.90, 1.5, ["Royal", "Dr. Oetker"], [("Pacote 1 kg", "PCT", 1)], (360, 540)),
        item("Mistura para Pudim de Baunilha", 29.90, 1, ["Dr. Oetker", "Royal"], [("Pacote 1 kg", "PCT", 1)], (360, 540)),
        item("Chocolate em Pó 50% Cacau", 49.90, 0.8, ["Nestlé", "Garoto", "Melken"], [("Pacote 1 kg", "PCT", 1)], (300, 540)),
        item("Cobertura Fracionada Sabor Chocolate", 34.90, 1, ["Harald", "Sicao"], [("Barra 1,01 kg", "UN", 1.01)], (300, 450)),
        item("Granulado de Chocolate", 27.90, 0.5, ["Harald", "Mavalério"], [("Pacote 1 kg", "PCT", 1)], (300, 450)),
        item("Geleia de Morango", 29.90, 1, ["Queensberry", "Predilecta"], [("Pote 1,01 kg", "PT", 1.01)], (240, 360)),
        item("Sorvete Sabor Creme", 18.90, 3, ["Kibon", "Nestlé"], [("Pote 10 L", "PT", 10)], (240, 365)),
        item("Paçoca de Rolha", 0.45, 20, ["Paçoquita"], [("Pote com 50 unidades", "PT", 50)], (180, 270)),
        item("Creme de Avelã com Cacau", 59.90, 0.3, ["Nutella"], [("Pote 3 kg", "PT", 3)], (240, 360)),
        item("Açúcar de Confeiteiro", 12.90, 0.5, ["União"], [("Pacote 1 kg", "PCT", 1)], (360, 540)),
    ]},
    "beverages": {"label": "Bebidas", "category": "Categoria Bebida", "target": 40, "items": [
        item("Café Torrado e Moído Tradicional", 54.90, 2.2, ["Três Corações", "Pilão", "Melitta", "Santa Clara"], COFFEE, (180, 360)),
        item("Café Torrado e Moído Extraforte", 56.90, 1.2, ["Três Corações", "Pilão"], COFFEE, (180, 360)),
        item("Café em Grãos Gourmet", 79.90, 0.4, ["Três Corações", "Melitta"], [("Pacote 1 kg", "PCT", 1)], (240, 360)),
        item("Achocolatado em Pó", 21.90, 2, ["Nescau", "Toddy"], [("Pacote 2 kg", "PCT", 2)], (300, 540)),
        item("Água Mineral sem Gás", 1.60, 60, ["Crystal", "Indaiá", "Minalba"], [("Fardo 12 × 500 ml", "FD", 6), ("Galão 20 L", "GL", 20)], (240, 360)),
        item("Suco de Laranja Integral", 11.90, 8, ["Del Valle", "Natural One"], [("Caixa 6 × 1 L", "CX", 6)], (120, 180)),
        item("Suco Concentrado de Caju", 9.90, 4, ["Maguary", "Dafruta"], [("Caixa 12 × 500 ml", "CX", 6)], (240, 360)),
        item("Suco Concentrado de Uva", 14.90, 3, ["Maguary", "Aurora"], [("Caixa 6 × 1,5 L", "CX", 9)], (240, 360)),
        item("Suco Concentrado de Maracujá", 13.90, 3, ["Maguary", "Dafruta"], [("Caixa 12 × 500 ml", "CX", 6)], (240, 360)),
        item("Refrigerante Cola", 5.40, 12, ["Coca-Cola", "Pepsi"], SODA, (150, 240)),
        item("Refrigerante Guaraná", 4.90, 10, ["Guaraná Antarctica", "Kuat"], SODA, (150, 240)),
        item("Chá Mate Tostado Natural", 79.90, 0.3, ["Matte Leão"], [("Caixa 250 g", "CX", 0.25)], (360, 540)),
        item("Água de Coco", 12.90, 3, ["Kero Coco", "Obrigado"], [("Caixa 12 × 1 L", "CX", 12)], (240, 360)),
        item("Leite de Coco", 21.90, 1, ["Sococo", "Menina"], [("Caixa 12 × 500 ml", "CX", 6)], (300, 420)),
    ]},
    "seasoning": {"label": "Temperos, Molhos e Condimentos", "category": None, "target": 28, "items": [
        item("Sal Refinado Iodado", 2.40, 4, ["Cisne", "Lebre"], [("Fardo 30 × 1 kg", "FD", 30)], (720, 1080)),
        item("Sal Grosso", 3.20, 0.8, ["Cisne"], [("Pacote 1 kg", "PCT", 1)], (720, 1080)),
        item("Açúcar Refinado", 5.40, 6, ["União", "Caravelas", "Guarani"], [("Fardo 10 × 1 kg", "FD", 10)], (540, 720)),
        item("Açúcar Cristal", 4.60, 8, ["União", "Guarani", "Caravelas"], [("Saco 50 kg", "SC", 50), ("Fardo 10 × 1 kg", "FD", 10)], (540, 720)),
        item("Pimenta-do-Reino Moída", 79.90, 0.15, ["Kitano", "BR Spices"], [("Pacote 500 g", "PCT", 0.5)], (360, 540)),
        item("Colorau", 19.90, 0.2, ["Kitano"], [("Pacote 1 kg", "PCT", 1)], (360, 540)),
        item("Cominho Moído", 49.90, 0.08, ["Kitano", "BR Spices"], [("Pacote 500 g", "PCT", 0.5)], (360, 540)),
        item("Orégano Desidratado", 69.90, 0.1, ["Kitano", "BR Spices"], [("Pacote 500 g", "PCT", 0.5)], (360, 540)),
        item("Páprica Doce", 54.90, 0.05, ["Kitano", "BR Spices"], [("Pacote 500 g", "PCT", 0.5)], (360, 540)),
        item("Louro em Folhas", 59.90, 0.02, ["Kitano"], [("Pacote 100 g", "PCT", 0.1)], (360, 540)),
        item("Caldo de Galinha", 39.90, 0.6, ["Knorr", "Maggi"], [("Pote 1,01 kg", "PT", 1.01)], (360, 540)),
        item("Caldo de Carne", 39.90, 0.5, ["Knorr", "Maggi"], [("Pote 1,01 kg", "PT", 1.01)], (360, 540)),
        item("Tempero Completo com Pimenta", 18.90, 1, ["Arisco"], [("Pote 3 kg", "PT", 3)], (360, 540)),
        item("Molho de Tomate Tradicional", 9.90, 5, ["Pomarola", "Heinz", "Fugini"], [("Bag 2 kg", "UN", 2)], (240, 360)),
        item("Extrato de Tomate", 14.90, 2, ["Elefante", "Quero"], [("Lata 1,7 kg", "LT", 1.7)], (540, 720)),
        item("Ketchup Tradicional", 15.90, 1, ["Heinz", "Hellmann's"], [("Bombona 3,5 kg", "BB", 3.5)], (240, 360)),
        item("Maionese Tradicional", 17.90, 1.5, ["Hellmann's", "Heinz"], [("Balde 3 kg", "BD", 3)], (180, 270)),
        item("Mostarda Amarela", 16.90, 0.5, ["Heinz", "Hemmer"], [("Bisnaga 1,01 kg", "BIS", 1.01)], (240, 360)),
        item("Vinagre de Álcool", 3.40, 3, ["Castelo", "Minhoto"], GALLON_5L, (720, 1080)),
        item("Molho Shoyu Tradicional", 14.90, 1, ["Sakura", "Kikkoman"], GALLON_5L, (540, 720)),
        item("Molho Inglês", 19.90, 0.2, ["Sakura"], [("Garrafa 1 L", "GF", 1)], (540, 720)),
    ]},
    "oils": {"label": "Óleos e Gorduras", "category": None, "target": 10, "items": [
        item("Óleo de Soja Refinado", 8.40, 9, ["Liza", "Soya", "Coamo"], [("Caixa 20 × 900 ml", "CX", 18), ("Bombona 18 L", "BB", 18)], (300, 450)),
        item("Óleo de Girassol", 14.90, 1, ["Liza"], [("Caixa 20 × 900 ml", "CX", 18)], (300, 450)),
        item("Óleo de Canola", 13.90, 1, ["Liza", "Salada"], [("Caixa 20 × 900 ml", "CX", 18)], (300, 450)),
        item("Azeite de Oliva Extravirgem", 52.90, 0.3, ["Gallo", "Andorinha"], [("Garrafa 500 ml", "GF", 0.5), ("Lata 3 L", "LT", 3)], (540, 720)),
        item("Gordura Vegetal Hidrogenada", 13.90, 1, ["Coamo"], [("Caixa 24 kg", "CX", 24)], (240, 360)),
    ]},
    "bakery": {"label": "Panificação", "category": None, "target": 14, "items": [
        item("Pão Francês", 0.85, 600, [None], [("Unidade 50 g", "UN", 1)], (1, 2)),
        item("Pão de Forma Tradicional", 9.90, 15, ["Pullman", "Wickbold", "Seven Boys"], [("Pacote 500 g", "PCT", 1)], (8, 12)),
        item("Pão de Hambúrguer", 9.90, 10, ["Pullman", "Wickbold"], [("Pacote com 8 unidades", "PCT", 1)], (8, 12)),
        item("Pão de Hot Dog", 8.90, 6, ["Pullman", "Seven Boys"], [("Pacote com 8 unidades", "PCT", 1)], (8, 12)),
        item("Bisnaguinha", 8.90, 6, ["Seven Boys", "Pullman"], [("Pacote 300 g", "PCT", 1)], (8, 12)),
        item("Biscoito Cream Cracker", 3.90, 20, ["Marilan", "Piraquê", "Vitarella"], [("Caixa com 20 pacotes", "CX", 20)], (180, 270)),
        item("Biscoito Maisena", 3.60, 10, ["Marilan", "Piraquê"], [("Caixa com 20 pacotes", "CX", 20)], (180, 270)),
        item("Torrada Tradicional", 6.90, 4, ["Bauducco", "Marilan"], [("Caixa com 20 pacotes", "CX", 20)], (180, 270)),
        item("Bolo Pronto Sabor Laranja", 11.90, 5, ["Bauducco", "Seven Boys"], [("Unidade 250 g", "UN", 1)], (60, 90)),
        item("Pão de Queijo Congelado", 24.90, 6, ["Forno de Minas"], [("Pacote 1 kg", "PCT", 1), ("Caixa 5 kg", "CX", 5)], (240, 365)),
        item("Massa para Pastel em Rolo", 16.90, 2, ["Massa Leve"], [("Rolo 1 kg", "RL", 1)], (60, 90)),
    ]},
    "cleaning": {"label": "Higiene e Limpeza", "category": None, "target": 16, "items": [
        item("Detergente Líquido Neutro", 5.00, 2.5, ["Ypê", "Minuano", "Limpol"], [("Caixa 24 × 500 ml", "CX", 12), ("Galão 5 L", "GL", 5)], (720, 1080)),
        item("Desinfetante Lavanda", 6.90, 2, ["Pinho Sol", "Veja", "Ypê"], GALLON_5L, (720, 1080)),
        item("Água Sanitária 2,5%", 3.20, 3, ["Qboa", "Ypê", "Candura"], GALLON_5L, (150, 180)),
        item("Álcool Líquido 70%", 9.90, 1.5, ["Itajá", "Tupi"], GALLON_5L, (720, 1080)),
        item("Sabão em Pó", 12.90, 1, ["Omo", "Ypê", "Minuano"], [("Pacote 5 kg", "PCT", 5)], (720, 1080)),
        item("Desengordurante para Cozinha Industrial", 14.90, 1, ["Veja", "Suma"], GALLON_5L, (720, 1080)),
        item("Limpador Multiuso", 7.90, 1, ["Veja", "Ypê"], GALLON_5L, (720, 1080)),
        item("Sanitizante para Hortifrúti", 29.90, 0.3, ["Hidrosteril"], [("Frasco 1 L", "FR", 1)], (360, 540)),
        item("Esponja Dupla Face", 1.40, 6, ["Scotch-Brite", "Bettanin"], [("Pacote com 4 unidades", "PCT", 4)]),
        item("Esponja de Lã de Aço", 0.55, 4, ["Bombril", "Assolan"], [("Pacote com 8 unidades", "PCT", 8)]),
        item("Pano Multiuso", 0.70, 6, ["Scotch-Brite", "Perfex"], [("Rolo com 300 panos", "RL", 300)]),
        item("Saco de Lixo Preto 100 L", 0.45, 20, ["Dover Roll", "Embalixo"], [("Pacote com 100 unidades", "PCT", 100)]),
    ]},
    "disposables": {"label": "Descartáveis e Embalagens", "category": None, "target": 12, "items": [
        item("Copo Descartável 200 ml", 0.07, 1200, ["Copobras", "Copaza", "Totalplast"], [("Caixa com 2.500 unidades", "CX", 2500)]),
        item("Copo Descartável 50 ml para Café", 0.04, 600, ["Copobras", "Copaza"], [("Caixa com 5.000 unidades", "CX", 5000)]),
        item("Guardanapo de Papel 24 × 24 cm", 0.03, 1500, ["Snob", "Scott"], [("Fardo com 2.400 unidades", "FD", 2400)]),
        item("Papel Toalha Interfolhado", 0.025, 1200, ["Santher", "Scott"], [("Fardo com 1.000 folhas", "FD", 1000)]),
        item("Filme PVC 28 cm × 300 m", 39.90, 0.5, ["Wyda", "Rolopac"], [("Rolo", "RL", 1)]),
        item("Papel Alumínio 30 cm × 100 m", 79.90, 0.2, ["Wyda", "Alumifort"], [("Rolo", "RL", 1)]),
        item("Luva Nitrílica Descartável", 0.35, 60, ["Supermax", "Descarpack"], [("Caixa com 100 unidades", "CX", 100)]),
        item("Touca Descartável Sanfonada", 0.12, 40, ["Descarpack", "Nobre"], [("Pacote com 100 unidades", "PCT", 100)]),
        item("Embalagem de Alumínio para Marmitex Nº 8", 0.65, 30, ["Wyda", "Alumifort"], [("Caixa com 100 unidades", "CX", 100)]),
        item("Pote Plástico com Tampa 500 ml", 0.55, 15, ["Prafesta", "Galvanotek"], [("Pacote com 25 unidades", "PCT", 25)]),
        item("Garfo Descartável para Refeição", 0.06, 300, ["Copobras", "Prafesta"], [("Pacote com 50 unidades", "PCT", 50)]),
    ]},
}

UNBRANDED_WITHOUT_BARCODE = {"produce_veg", "produce_fruit", "bakery"}

total_target = sum(group["target"] for group in FOOD_GROUPS.values())
if total_target != ROWS_PER_TABLE:
    raise RuntimeError(f"O catálogo soma {total_target} produtos; o esperado é {ROWS_PER_TABLE}.")

print(f"Catálogo de referência carregado: {sum(len(g['items']) for g in FOOD_GROUPS.values())} itens em {len(FOOD_GROUPS)} grupos")

## 6.1 Grupos de alimento na árvore de categorias (V019)

Cada grupo de alimento com categoria oficial vira uma categoria **filha** dela, e o produto aponta para o grupo (a categoria oficial é a raiz). O `vw_category_tree` percorre a árvore com `WITH RECURSIVE`.

In [ ]:
group_rows = [
    (group["label"], f"{group['label']}, grupo de alimento da {group['category']}.", category_id_by_name[group["category"]])
    for group in FOOD_GROUPS.values() if group["category"]
]

result = execute_values(
    cursor,
    "INSERT INTO tb_category (name, description, id_parent_category) VALUES %s RETURNING id_category, name",
    group_rows,
    fetch=True,
)
group_category_id = {name: id_category for id_category, name in result}
category_id_by_group = {
    key: group_category_id[group["label"]] for key, group in FOOD_GROUPS.items() if group["category"]
}

conn.commit()
print(f"{len(group_category_id)} categorias de grupo de alimento criadas sob as {len(category_id_by_name)} oficiais")

cursor.execute("SELECT category_level, COUNT(*) FROM vw_category_tree GROUP BY category_level ORDER BY category_level")
for level, qtd in cursor.fetchall():
    print(f"   nível {level}: {qtd}")

## 6.2 Produtos reais do Open Food Facts

Parte do catálogo vem da [Open Food Facts](https://world.openfoodfacts.org), a mesma base pública que a API usa na leitura de código de barras (`OpenFoodFactsClient`). Para cada marca do catálogo, o seed busca produtos vendidos no Brasil em `search.openfoodfacts.org` (o motor de busca oficial da Open Food Facts) e aproveita os que têm EAN brasileiro válido, foto e quantidade. Esses produtos entram com **nome, marca, código de barras e URL da foto reais** (`photo_url`); preço, consumo e validade vêm do item equivalente do catálogo, proporcionais ao tamanho da embalagem.

- O resultado da busca fica em cache por 7 dias em `openfoodfacts_cache.json` (ao lado do notebook), para não repetir as consultas a cada execução.
- Se a Open Food Facts estiver fora do ar, o seed segue só com o catálogo curado (sem foto) e avisa.
- Dados de produtos © colaboradores da Open Food Facts, licença ODbL; imagens sob CC BY-SA.

In [ ]:
OFF_SEARCH_URL = "https://search.openfoodfacts.org/search"
OFF_USER_AGENT = "InventraSeed/1.0 (projeto academico Inventra)"  # a Open Food Facts pede um User-Agent que identifique o app
OFF_FIELDS = "code,product_name,product_name_pt,brands,quantity,image_front_url,image_url"
OFF_SHARE = 0.35          # até 35% de cada grupo vem da Open Food Facts; o resto é o catálogo curado
OFF_MIN_AMOUNT = 0.25     # embalagens menores que 250 g/ml não fazem sentido no estoque de uma cozinha industrial
OFF_CACHE = Path("openfoodfacts_cache.json")
OFF_CACHE_DAYS = 7

QTY_RE = re.compile(r"(?:(\d+)\s*[×x*]\s*)?(\d+(?:[.,]\d+)?)\s*(kg|g|gr|l|lt|litros?|ml)\b", re.I)
UNIT_FACTOR = {"kg": 1, "g": 0.001, "gr": 0.001, "l": 1, "lt": 1, "litro": 1, "litros": 1, "ml": 0.001}
OFF_STOPWORDS = {"de", "da", "do", "das", "dos", "e", "em", "com", "sem", "para", "ao", "a", "o", "tipo", "n", "no", "na"}
# outro tipo de produto que só cita o item no nome (ex.: "Pizza de Lombo", "Hambúrguer de Carne Bovina")
OFF_EXCLUDED_WORDS = {"pizza", "hamburguer", "burger", "lasanha", "sanduiche", "lanche", "hot", "pate", "sopa", "pudim", "bolo",
                      "empada", "kit"}
ENGLISH_NAME = re.compile(r"\b(chicken|breast|milk|cheese|juice|sauce|rice|beans|coffee|cream|sugar|flour|bread|meat)\b", re.I)
ABBREVIATION = re.compile(r"\b[A-Za-zÀ-ú]{2,5}\.")
LOWER_WORDS = {"de", "da", "do", "das", "dos", "e", "em", "com", "sem", "para", "ao", "à", "a", "o", "ou", "no", "na"}
# grafias sem acento ou siglas em minúsculas que aparecem nos nomes cadastrados por colaboradores da Open Food Facts
WORD_FIXES = {"cafe": "Café", "feijao": "Feijão", "pao": "Pão", "acucar": "Açúcar", "agua": "Água", "macarrao": "Macarrão",
              "oleo": "Óleo", "file": "Filé", "graos": "Grãos", "coracoes": "Corações", "uht": "UHT", "iqf": "IQF"}
# outra forma de escrever a marca no nome: com ela presente, a marca não é repetida no fim
BRAND_ALIASES = {"Três Corações": ["3coracoes"], "Coca-Cola": ["cocacola"], "Matte Leão": ["matteleao", "leao"]}


def parse_amount(text):
    """Quantidade total em kg ou litros ("395 g" -> 0.395, "27 × 395 g" -> 10.665); None se não houver."""
    match = QTY_RE.search(text or "")
    if not match:
        return None
    multiplier = int(match.group(1)) if match.group(1) else 1
    return multiplier * float(match.group(2).replace(",", ".")) * UNIT_FACTOR[match.group(3).lower()]


def words(text):
    return [w for w in slugify(text, " ").split() if w not in OFF_STOPWORDS and not w.isdigit()]


def priced_by_amount(it):
    """Item com preço e consumo por kg/litro: só ele pode receber um produto real de outro tamanho."""
    return any(unit in MEASURE_UNITS or (parse_amount(label) and abs(parse_amount(label) - content) <= 0.05 * content)
               for label, unit, content in it["packs"])


def valid_ean(code):
    if not re.fullmatch(r"(789|790)\d{10}", code):
        return False
    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(code[:12]))
    return (10 - total % 10) % 10 == int(code[12])


def clean_name(name, brand, quantity):
    """Padroniza o nome da Open Food Facts: maiúsculas de título, marca e quantidade sempre presentes."""
    tokens = re.sub(r"\s+", " ", name).strip(" -").split(" ")
    out = []
    for i, token in enumerate(tokens):
        low = token.lower()
        if slugify(token, "") == slugify(brand, ""):
            out.append(brand)
        elif low in WORD_FIXES:
            out.append(WORD_FIXES[low])
        elif i and low in LOWER_WORDS:
            out.append(low)
        elif any(ch.isdigit() for ch in token):
            out.append(low)
        elif token.isupper() and len(token) <= 4:
            out.append(token)  # siglas: UHT, IQF
        else:
            out.append(token[:1].upper() + token[1:].lower())
    if out.count(brand) > 1:  # "Liza Óleo de Canola Liza" -> "Óleo de Canola Liza"
        out.remove(brand)
    text = " ".join(out)
    if not any(alias in slugify(text, "") for alias in [slugify(brand, "")] + BRAND_ALIASES.get(brand, [])):
        text += f" {brand}"
    if not QTY_RE.search(text):
        text += f" {quantity.strip()}"
    text = re.sub(r"(\d)\s*(kg|g|ml|l)\b", lambda m: f"{m.group(1)} {'L' if m.group(2).lower() == 'l' else m.group(2).lower()}",
                  text, flags=re.I)
    return text[:150]


def off_search(brand):
    query = f'brands_tags:"{slugify(brand, "-")}" countries_tags:"en:brazil"'
    url = OFF_SEARCH_URL + "?" + urllib.parse.urlencode({"q": query, "page_size": 100, "fields": OFF_FIELDS})
    request = urllib.request.Request(url, headers={"User-Agent": OFF_USER_AGENT})
    for attempt in range(3):
        try:
            with urllib.request.urlopen(request, timeout=20) as response:
                return brand, json.load(response).get("hits", [])
        except Exception:
            clock.sleep(2 * (attempt + 1))
    return brand, None


# marca -> itens do catálogo vendidos por ela (só os precificados por kg/litro)
items_by_brand = defaultdict(list)
for group_key, group in FOOD_GROUPS.items():
    for it in group["items"]:
        if priced_by_amount(it):
            for brand in it["brands"]:
                if brand:
                    items_by_brand[brand].append((group_key, it))

off_hits = {}
if OFF_CACHE.exists() and clock.time() - OFF_CACHE.stat().st_mtime < OFF_CACHE_DAYS * 86400:
    off_hits = json.loads(OFF_CACHE.read_text(encoding="utf-8"))
missing = [b for b in items_by_brand if b not in off_hits]
if missing:
    print(f"Consultando a Open Food Facts para {len(missing)} marcas...")
    with ThreadPoolExecutor(max_workers=4) as pool:
        for brand, hits in pool.map(off_search, missing):
            if hits is not None:
                off_hits[brand] = hits
    if off_hits:
        OFF_CACHE.write_text(json.dumps(off_hits, ensure_ascii=False), encoding="utf-8")


def best_item(name, brand):
    """Item do catálogo que corresponde ao produto: o nome do item aparece no começo do nome do produto."""
    name_words = words(name)
    best, best_score = None, 0
    for group_key, it in items_by_brand[brand]:
        item_words = words(it["name"])
        if item_words[0] not in name_words[:3]:
            continue
        score = len(set(item_words) & set(name_words))
        if score >= min(2, len(item_words)) and score > best_score:
            best, best_score = (group_key, it), score
    return best


off_by_group = defaultdict(list)
seen_codes, seen_names = set(), set()
for brand in sorted(off_hits):
    for hit in off_hits[brand]:
        raw_name = (hit.get("product_name_pt") or hit.get("product_name") or "").strip()
        code = str(hit.get("code", ""))
        photo_url = hit.get("image_front_url") or hit.get("image_url") or ""
        quantity = hit.get("quantity") or ""
        amount = parse_amount(quantity)
        if not (raw_name and valid_ean(code) and photo_url.startswith("https://") and len(photo_url) <= 255):
            continue
        if amount is None or amount < OFF_MIN_AMOUNT or code in seen_codes:
            continue
        if raw_name.isupper() or ENGLISH_NAME.search(raw_name) or ABBREVIATION.search(raw_name):
            continue
        if OFF_EXCLUDED_WORDS & set(words(raw_name)):
            continue
        match = best_item(raw_name, brand)
        if not match:
            continue
        name = clean_name(raw_name, brand, quantity)
        if slugify(name, "") in seen_names:
            continue
        seen_codes.add(code)
        seen_names.add(slugify(name, ""))
        group_key, it = match
        off_by_group[group_key].append({"name": name, "brand": brand, "barcode": code, "photo_url": photo_url,
                                        "item": it, "amount": amount})

off_total = sum(len(v) for v in off_by_group.values())
if off_hits:
    print(f"{off_total} produtos reais aproveitáveis da Open Food Facts ({len(off_hits)} marcas consultadas)")
else:
    print("AVISO: Open Food Facts indisponível e sem cache; o catálogo segue só com os produtos curados.")

## 6.3 Cadastro dos produtos

Cada grupo recebe até 35% de produtos reais da Open Food Facts (variando o item, para não repetir o mesmo produto em vários tamanhos) e completa a meta com o catálogo curado: item + marca + embalagem de compra (ex.: *Picanha Bovina Resfriada Friboi — Peça a vácuo*). Os produtos da Open Food Facts são controlados por unidade (a embalagem); os curados, na unidade de compra do item (kg, caixa, fardo...).

In [ ]:
def pick_diverse(candidates, n):
    """n produtos alternando entre itens do catálogo, para não pegar 5 tamanhos do mesmo café."""
    by_item = defaultdict(list)
    for candidate in candidates:
        by_item[candidate["item"]["name"]].append(candidate)
    queues = list(by_item.values())
    random.shuffle(queues)
    for queue in queues:
        random.shuffle(queue)
    picked = []
    while len(picked) < n and queues:
        for queue in list(queues):
            if len(picked) == n:
                break
            picked.append(queue.pop())
            if not queue:
                queues.remove(queue)
    return picked


def product_created_at():
    # 85% do catálogo é antigo; o resto entrou nos últimos meses (alguns vieram de leitura de
    # código de barras não cadastrado, ver alertas UNREGISTERED_PRODUCT)
    if random.random() < 0.85:
        return work_datetime(date(2023, 1, 9) + timedelta(days=random.randint(0, 1050)))
    return past_work_datetime(20, 150)


rows = []
products = []
used_barcodes = {c["barcode"] for group in off_by_group.values() for c in group}
for group_key, group in FOOD_GROUPS.items():
    id_category = category_id_by_group.get(group_key)
    real = pick_diverse(off_by_group[group_key], min(len(off_by_group[group_key]), round(group["target"] * OFF_SHARE)))
    for candidate in real:
        it, amount = candidate["item"], candidate["amount"]
        created_at, active = product_created_at(), random.random() < 0.97
        rows.append((candidate["name"], candidate["brand"], id_category, unit_id_by_symbol["UN"], candidate["barcode"],
                     candidate["photo_url"], active, created_at))
        products.append({
            "name": candidate["name"], "group": group_key, "brand": candidate["brand"], "unit": "UN",
            "price": round(it["price"] * amount * random.uniform(0.95, 1.08), 2),
            "use": it["use"] / amount, "life": it["life"], "barcode": candidate["barcode"],
            "active": active, "created_at": created_at, "source": "Open Food Facts",
        })

    combos = [(it, brand, pack) for it in group["items"] for brand in it["brands"] for pack in it["packs"]]
    curated_target = group["target"] - len(real)
    if len(combos) < curated_target:
        raise RuntimeError(f"{group_key}: só {len(combos)} combinações para {curated_target} produtos")

    for it, brand, (pack_label, unit, content) in random.sample(combos, curated_target):
        name = f"{it['name']} {brand} — {pack_label}" if brand else f"{it['name']} — {pack_label}"

        barcode = None
        if not (brand is None and group_key in UNBRANDED_WITHOUT_BARCODE):
            barcode = ean13()
            while barcode in used_barcodes:
                barcode = ean13()
            used_barcodes.add(barcode)

        created_at, active = product_created_at(), random.random() < 0.97
        rows.append((name, brand, id_category, unit_id_by_symbol[unit], barcode, None, active, created_at))
        products.append({
            "name": name, "group": group_key, "brand": brand, "unit": unit,
            "price": round(it["price"] * content, 2),  # R$ por unidade de estoque
            "use": it["use"] / content,                 # unidades de estoque por dia a cada 1.000 refeições
            "life": it["life"], "barcode": barcode, "active": active, "created_at": created_at,
            "source": "catálogo",
        })

result = execute_values(
    cursor,
    """INSERT INTO tb_product (name, brand, id_category, id_unit, barcode, photo_url, active, created_at)
       VALUES %s RETURNING id_product""",
    rows,
    fetch=True,
)
for product, (id_product,) in zip(products, result):
    product["id"] = id_product

conn.commit()
from_off = sum(1 for p in products if p["source"] == "Open Food Facts")
without_category = sum(1 for r in rows if r[2] is None)
print(f"{len(products)} produtos criados: {from_off} reais da Open Food Facts (com foto) e {len(products) - from_off} do catálogo curado; "
      f"{without_category} sem categoria (temperos, óleos, panificação, limpeza e descartáveis)")

## 6.4 Mix de produtos de cada cozinha

Cada cozinha trabalha com um mix fixo de produtos (proteínas, hortifrúti, grãos, laticínios, temperos, bebidas e limpeza). Esse mix é a base dos parâmetros de estoque (seção 13), dos lotes (seção 11) e das requisições (seção 9). São 500 pares produto x cozinha no total.

In [ ]:
MIX_SLOTS = [
    ("beef",), ("beef",), ("poultry",), ("pork", "fish"),
    ("dairy",), ("produce_veg",), ("produce_veg",), ("produce_fruit",),
    ("grains",), ("grains",), ("seasoning", "oils"), ("beverages", "sweets", "bakery"),
]
EXTRA_SLOT = ("cleaning", "disposables")

active_by_group = defaultdict(list)
for idx, product in enumerate(products):
    if product["active"]:
        active_by_group[product["group"]].append(idx)

# metade das cozinhas tem 13 itens no mix e a outra metade 12: 500 pares no total
extra_slots = ROWS_PER_TABLE - len(MIX_SLOTS) * len(kitchens)
kitchens_with_extra = set(random.sample(range(len(kitchens)), extra_slots))

kitchen_mix = defaultdict(list)   # índice da cozinha -> índices de produto
pair_daily_use = {}               # (cozinha, produto) -> consumo médio diário na unidade de estoque
for k_idx, kitchen in enumerate(kitchens):
    slots = MIX_SLOTS + ([EXTRA_SLOT] if k_idx in kitchens_with_extra else [])
    for slot in slots:
        candidates = [p for g in slot for p in active_by_group[g] if p not in kitchen_mix[k_idx]]
        # itens de alto giro (arroz, frango, alface...) aparecem em mais cozinhas
        p_idx = random.choices(candidates, weights=[products[p]["use"] * products[p]["price"] ** 0.5 + 1 for p in candidates])[0]
        kitchen_mix[k_idx].append(p_idx)
        pair_daily_use[(k_idx, p_idx)] = products[p_idx]["use"] * kitchen["meals"] / 1000 * random.uniform(0.6, 1.4)

mix_pairs = list(pair_daily_use)
print(f"{len(mix_pairs)} pares produto x cozinha no mix ({len({p for _, p in mix_pairs})} produtos distintos em uso)")

# 7. USUÁRIOS (tb_user)

12 colaboradores por cozinha em média, com o perfil de cada um: 1 ou 2 supervisores, compradores e estoquistas. 20 usuários recém-cadastrados ainda não têm cozinha: pediram para entrar em uma e aguardam o supervisor (alertas ACCESS_REQUEST, seção 16). Parte da equipe entra com a conta Google (sem senha, com `firebase_uid`).

In [ ]:
KITCHENLESS_USERS = 20
BCRYPT_ALPHABET = "./ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
FIREBASE_ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
HISTORY_DAYS = 21  # janela do histórico de acessos diários (seção 18)

# equipe proporcional ao porte da cozinha (refeições/dia), mínimo de 7 pessoas
staff_total = ROWS_PER_TABLE - KITCHENLESS_USERS
total_meals = sum(k["meals"] for k in kitchens)
headcount = [max(7, int(staff_total * k["meals"] / total_meals)) for k in kitchens]
while sum(headcount) > staff_total:
    headcount[headcount.index(max(headcount))] -= 1
while sum(headcount) < staff_total:
    headcount[random.randrange(len(kitchens))] += 1

used_emails = set()


def make_identity(domain):
    first = fake.first_name()
    last = fake.last_name()
    name = f"{first} {last}" if random.random() < 0.5 else f"{first} {fake.last_name()} {last}"
    local = f"{slugify(first)}.{slugify(last)}"
    email = f"{local}@{domain}"
    n = 2
    while email in used_emails:
        email = f"{local}{n}@{domain}"
        n += 1
    used_emails.add(email)
    return name, email


def recent_login_day():
    # dias úteis pesam mais: as cozinhas rodam todos os dias, mas o escritório não
    while True:
        day = TODAY - timedelta(days=random.randint(0, HISTORY_DAYS - 1))
        if day.weekday() < 5 or random.random() < 0.5:
            return day


def login_at(day, not_before):
    dt = datetime.combine(day, time(5, 30)) + timedelta(minutes=random.randint(0, 16 * 60))
    dt = min(dt, NOW - timedelta(minutes=random.randint(1, 120)))
    return max(dt, not_before + timedelta(minutes=10))


users = []
rows = []
plan = []
for k_idx, count in enumerate(headcount):
    supervisors = 2 if count >= 14 else 1
    buyers = max(2, round(count * 0.25))
    plan += [(k_idx, "supervisor")] * supervisors + [(k_idx, "comprador")] * buyers
    plan += [(k_idx, "estoquista")] * (count - supervisors - buyers)
plan += [(None, random.choice(["estoquista", "estoquista", "comprador"])) for _ in range(KITCHENLESS_USERS)]

for k_idx, profile in plan:
    if k_idx is None:
        domain = random.choice(list(COMPANIES.values()))["domain"]
        created_at = past_work_datetime(2, 20)
        active = True
    else:
        kitchen = kitchens[k_idx]
        domain = kitchen["domain"]
        start = max(kitchen["created_at"], datetime(2022, 6, 1))
        # supervisores costumam estar na cozinha desde a implantação
        if profile == "supervisor":
            created_at = rand_datetime(start, start + timedelta(days=60))
        else:
            created_at = rand_datetime(start, NOW - timedelta(days=30))
        # todo supervisor está ativo: é ele quem aprova as requisições da cozinha
        active = profile == "supervisor" or random.random() < 0.96
    name, email = make_identity(domain)

    if active:
        last_login = login_at(recent_login_day() if k_idx is not None else TODAY - timedelta(days=random.randint(0, 1)), created_at)
    else:
        last_login = rand_datetime(created_at, max(created_at, NOW - timedelta(days=45)))

    google = random.random() < 0.2
    password_hash = None if google else "$2a$10$" + "".join(random.choices(BCRYPT_ALPHABET, k=53))
    firebase_uid = "".join(random.choices(FIREBASE_ALPHABET, k=28)) if google else None
    id_user = fake.uuid4()

    rows.append((
        id_user, name, email, password_hash, firebase_uid,
        kitchens[k_idx]["id"] if k_idx is not None else None, profile_id_by_code[profile],
        active, last_login, created_at,
    ))
    users.append({"id": id_user, "name": name, "kitchen": k_idx, "profile": profile, "active": active,
                  "created_at": created_at, "last_login": last_login})

execute_values(
    cursor,
    """INSERT INTO tb_user
        (id_user, name, email, password_hash, firebase_uid, id_kitchen, id_profile, active, last_login, created_at)
       VALUES %s""",
    rows,
)

staff = defaultdict(list)  # (cozinha, perfil) -> usuários ativos
for user in users:
    if user["kitchen"] is not None and user["active"]:
        staff[(user["kitchen"], user["profile"])].append(user)


def pick_staff(k_idx, profiles, before=None):
    """Colaborador ativo da cozinha, no primeiro perfil da lista que tiver alguém cadastrado antes da data."""
    for profile in profiles:
        pool = [u for u in staff[(k_idx, profile)] if before is None or u["created_at"] < before]
        if pool:
            return random.choice(pool)
    return random.choice([u for p in ("supervisor", "estoquista", "comprador") for u in staff[(k_idx, p)]])


conn.commit()
by_profile = defaultdict(int)
for user in users:
    by_profile[user["profile"]] += 1
print(f"{len(users)} usuários criados: " + ", ".join(f"{n} {p}" for p, n in sorted(by_profile.items())))

# 8. PRODUTO x FORNECEDOR (tb_product_supplier)

Quem fornece cada produto: as marcas do grupo (Friboi, Maturatta, Seara, Swift) vêm da divisão interna correspondente; os demais produtos, de distribuidores do segmento certo (carne de frigorífico, alface de hortifrúti, detergente de distribuidora de limpeza). Todo produto do mix das cozinhas tem ao menos um fornecedor.

In [ ]:
internal_by_brand = {}
for supplier in suppliers:
    for brand in supplier["brands"]:
        internal_by_brand[brand] = supplier

suppliers_by_group = defaultdict(list)
for supplier in suppliers:
    if supplier["active"]:
        for group in supplier["groups"]:
            suppliers_by_group[group].append(supplier)


def candidate_suppliers(product):
    pool = list(suppliers_by_group[product["group"]])
    internal = internal_by_brand.get(product["brand"])
    if internal:
        # marca do grupo: a divisão interna é a fonte principal, distribuidor de carnes é a alternativa
        return [internal] + [s for s in pool if s["segment"] in ("frigorifico", "avicola", "atacado")]
    return pool


links_by_product = defaultdict(list)  # índice do produto -> [(fornecedor, preço de referência, prazo)]
used_pairs = set()
rows = []


def add_link(p_idx, supplier):
    if (p_idx, supplier["id"]) in used_pairs:
        return False
    used_pairs.add((p_idx, supplier["id"]))
    product = products[p_idx]
    reference_price = round(product["price"] * random.uniform(0.94, 1.08), 2)
    lead_time = random.randint(*supplier["lead"])
    code = f"{supplier['prefix']}-{random.randint(10000, 99999)}"
    rows.append((product["id"], supplier["id"], code, reference_price, lead_time))
    links_by_product[p_idx].append((supplier, reference_price, lead_time))
    return True


mix_products = sorted({p for _, p in mix_pairs})
for p_idx in mix_products:
    candidates = candidate_suppliers(products[p_idx])
    add_link(p_idx, candidates[0] if internal_by_brand.get(products[p_idx]["brand"]) else random.choice(candidates))

# segunda e terceira opção de compra, priorizando os produtos que as cozinhas usam
attempts = 0
while len(rows) < ROWS_PER_TABLE:
    attempts += 1
    if attempts > ROWS_PER_TABLE * 50:
        raise RuntimeError("Não foi possível completar os vínculos produto x fornecedor")
    p_idx = random.choice(mix_products) if random.random() < 0.6 else random.randrange(len(products))
    candidates = candidate_suppliers(products[p_idx])
    if candidates:
        add_link(p_idx, random.choice(candidates))

execute_values(
    cursor,
    """INSERT INTO tb_product_supplier
        (id_product, id_supplier, supplier_code, reference_price, lead_time_days)
       VALUES %s""",
    rows,
)

conn.commit()
print(f"{len(rows)} vínculos produto x fornecedor criados ({len(links_by_product)} produtos com fornecedor)")

# 9. REQUISIÇÕES (tb_requisition)

Pedidos de compra (comprador), de consumo para a produção (estoquista) e de transferência entre unidades, nos últimos 5 meses. Os status seguem o fluxo da API: rascunho → em análise → aprovada / recusada / cancelada. Quem aprova ou recusa é sempre um supervisor da própria cozinha; na recusa e no cancelamento, `reason` guarda o motivo informado (como fazem `sp_reject_requisition` e `sp_cancel_requisition`).

In [ ]:
REQUISITION_ORIGINS = {
    "CONSUMPTION": ["Cozinha Quente", "Pré-preparo", "Confeitaria", "Refeitório", "Açougue Interno", "Lanchonete"],
    "PURCHASE": ["Almoxarifado", "Estoque Seco", "Câmara Fria", "Compras", "Câmara de Congelados"],
}
REQUISITION_REASONS = {
    "PURCHASE": [
        "Reposição do estoque mínimo para o cardápio da semana {week}.",
        "Compra programada quinzenal de {group}.",
        "Reposição após o inventário cíclico da semana {week}.",
        "Aumento de demanda com o turno extra de produção.",
        "Antecipação de compra antes do reajuste de preço do fornecedor.",
        "Reposição de lote recusado no recebimento por avaria.",
    ],
    "CONSUMPTION": [
        "Abastecimento da linha de produção do almoço.",
        "Retirada para o jantar do turno C.",
        "Pré-preparo do cardápio da semana {week}.",
        "Abastecimento do refeitório para o café da manhã.",
        "Coffee break do treinamento de segurança (NR-12).",
        "Lanche da SIPAT para os colaboradores da unidade.",
    ],
    "TRANSFER": [
        "Transferência para cobrir ruptura de estoque na {dest}.",
        "Remanejamento de estoque excedente para a {dest}.",
        "Empréstimo de insumos para a {dest} até a chegada do pedido.",
    ],
}
REJECTION_REASONS = [
    "Quantidade acima do consumo médio da cozinha; revisar o pedido.",
    "Já existe pedido em aberto para o mesmo item.",
    "Fornecedor sugerido sem cadastro ativo; refazer com fornecedor homologado.",
    "Estoque atual cobre a demanda das próximas duas semanas.",
    "Orçamento do mês já atingido para esta categoria.",
]
CANCEL_REASONS = [
    "Item substituído no cardápio da semana.",
    "Pedido lançado em duplicidade.",
    "Fornecedor sem disponibilidade para a data necessária.",
    "Evento cancelado pela unidade.",
]

kitchen_weights = [k["meals"] for k in kitchens]
requisitions = []
rows = []
for _ in range(ROWS_PER_TABLE):
    k_idx = random.choices(range(len(kitchens)), weights=kitchen_weights)[0]
    kitchen = kitchens[k_idx]
    req_type = random.choices(["CONSUMPTION", "PURCHASE", "TRANSFER"], weights=[45, 40, 15])[0]
    status = random.choices(["DRAFT", "UNDER_REVIEW", "APPROVED", "REJECTED", "CANCELLED"], weights=[5, 9, 66, 11, 9])[0]

    if status == "DRAFT":
        created_at = past_work_datetime(0, 5)
    elif status == "UNDER_REVIEW":
        created_at = past_work_datetime(0, 4)
    else:
        # o volume de pedidos se concentra nas últimas semanas, com histórico de até 5 meses
        days_ago = min(150, 1 + int(random.expovariate(1 / 30)))
        created_at = past_work_datetime(days_ago, days_ago)

    requester = pick_staff(k_idx, ["comprador"] if req_type == "PURCHASE" else ["estoquista", "comprador"], created_at)
    supervisor = pick_staff(k_idx, ["supervisor"], created_at)

    if req_type == "TRANSFER":
        other = random.choice([k for k in kitchens if k is not kitchen])
        destination = f"Unidade {other['city']}"
        origin = destination if len(destination) <= 20 else other["city"][:20]
    else:
        destination = None
        origin = random.choice(REQUISITION_ORIGINS[req_type])

    group_label = random.choice([FOOD_GROUPS[products[p]["group"]]["label"].lower() for p in kitchen_mix[k_idx]])
    reason = random.choice(REQUISITION_REASONS[req_type]).format(
        week=created_at.isocalendar()[1], group=group_label, dest=f"unidade de {destination.split(' ', 1)[-1]}" if destination else "")

    id_approver, approved_at, decided_at = None, None, None
    if status == "APPROVED":
        id_approver = supervisor["id"]
        approved_at = min(created_at + timedelta(minutes=random.randint(40, 30 * 60)), NOW - timedelta(minutes=5))
        decided_at = approved_at
    elif status == "REJECTED":
        id_approver = supervisor["id"]
        decided_at = min(created_at + timedelta(minutes=random.randint(40, 30 * 60)), NOW - timedelta(minutes=5))
        reason = random.choice(REJECTION_REASONS)
    elif status == "CANCELLED":
        reason = random.choice(CANCEL_REASONS)

    rows.append((req_type, origin, status, reason, kitchen["id"], requester["id"], id_approver, created_at, approved_at))
    requisitions.append({"type": req_type, "status": status, "kitchen": k_idx, "requester": requester,
                         "approver": supervisor if id_approver else None, "created_at": created_at,
                         "decided_at": decided_at, "reason": reason, "items": []})

result = execute_values(
    cursor,
    """INSERT INTO tb_requisition
        (requisition_type, origin, status, reason, id_kitchen,
         id_requester_user, id_approver_user, created_at, approved_at)
       VALUES %s RETURNING id_requisition""",
    rows,
    fetch=True,
)
for req, (id_requisition,) in zip(requisitions, result):
    req["id"] = id_requisition

conn.commit()
by_status = defaultdict(int)
for req in requisitions:
    by_status[req["status"]] += 1
print(f"{len(requisitions)} requisições criadas: " + ", ".join(f"{n} {s}" for s, n in sorted(by_status.items())))

# 10. ITENS DE REQUISIÇÃO (tb_requisition_item)

Requisição enviada para análise precisa ter ao menos um item (regra da API); parte dos rascunhos ainda está vazia. Os itens saem do mix da cozinha, com quantidade proporcional ao consumo dela e preço estimado pelo preço de referência do fornecedor.

In [ ]:
ITEM_NOTES = {
    "PURCHASE": [
        "Validade mínima de 60 dias na entrega.",
        "Entregar até quinta-feira, antes do turno da manhã.",
        "Conferir temperatura no recebimento (até 4 °C).",
        "Preferência por embalagem a vácuo.",
        "Mesmo fornecedor do último pedido, se possível.",
    ],
    "CONSUMPTION": [
        "Separar até as 9h para o pré-preparo.",
        "Retirar do lote com vencimento mais próximo.",
        "Uso no cardápio especial de sexta-feira.",
    ],
    "TRANSFER": [
        "Transporte refrigerado agendado com a logística.",
        "Enviar com a nota de transferência entre filiais.",
    ],
}

needs_items = [r for r in requisitions if r["status"] != "DRAFT" or random.random() < 0.6]
item_counts = {id(r): 1 for r in needs_items}
extra = ROWS_PER_TABLE - len(needs_items)
if extra < 0:
    raise RuntimeError(f"{len(needs_items)} requisições precisam de item, mais que {ROWS_PER_TABLE}")
for req in random.choices([r for r in needs_items if r["status"] != "DRAFT"], k=extra):
    item_counts[id(req)] = min(item_counts[id(req)] + 1, len(kitchen_mix[req["kitchen"]]))
while sum(item_counts.values()) < ROWS_PER_TABLE:
    item_counts[id(random.choice(needs_items))] += 1

rows = []
for req in needs_items:
    k_idx = req["kitchen"]
    for p_idx in random.sample(kitchen_mix[k_idx], item_counts[id(req)]):
        product = products[p_idx]
        daily = pair_daily_use[(k_idx, p_idx)]
        days = {"PURCHASE": (7, 20), "CONSUMPTION": (0.5, 2), "TRANSFER": (2, 6)}[req["type"]]
        quantity = round_qty(daily * random.uniform(*days), product["unit"])
        links = links_by_product.get(p_idx, [])

        suggested, estimated_price = None, None
        if req["type"] == "PURCHASE":
            if links and random.random() < 0.85:
                supplier, reference_price, _ = random.choice(links)
                suggested = supplier["id"]
                estimated_price = round(reference_price * random.uniform(0.97, 1.06), 2)
            elif random.random() < 0.7:
                estimated_price = round(product["price"] * random.uniform(0.95, 1.1), 2)
        elif random.random() < 0.6:
            estimated_price = round(product["price"] * random.uniform(0.95, 1.05), 2)
        note = random.choice(ITEM_NOTES[req["type"]]) if random.random() < 0.25 else None

        rows.append((req["id"], product["id"], suggested, quantity, estimated_price, note))
        req["items"].append({"product": p_idx, "quantity": quantity})

execute_values(
    cursor,
    """INSERT INTO tb_requisition_item
        (id_requisition, id_product, id_suggested_supplier, quantity, estimated_price, note)
       VALUES %s""",
    rows,
)

conn.commit()
print(f"{len(rows)} itens de requisição criados ({sum(1 for r in requisitions if not r['items'])} rascunhos ainda sem item)")

# 11. LOTES DE ESTOQUE (tb_stock_batch)

Cada lote é uma entrada de mercadoria de um produto do mix da cozinha, com quantidade para alguns dias (perecíveis) ou semanas (secos e congelados) de consumo. O saldo atual é o que sobrou depois do consumo desde a entrada, e o status acompanha: `WRITTEN_OFF` quando acabou, `EXPIRED` quando venceu com saldo. Alguns pares do mix estão sem lote (ruptura de estoque).

Ao inserir lote vencido, a trigger `trg_expiration_alert` já cria o alerta EXPIRATION, como no sistema real.

In [ ]:
RUPTURE_PAIRS = 20      # pares do mix sem nenhum lote em estoque
LOW_STOCK_SHARE = 0.08  # pares com lote, mas com saldo abaixo do estoque mínimo
DEFAULT_LEAD_DAYS = 3   # prazo de entrega de produto sem fornecedor vinculado

# estoque mínimo = consumo durante o prazo do fornecedor mais rápido + 1 dia de segurança (usado na seção 13)
pair_min_days = {}
for k_idx, p_idx in mix_pairs:
    links = links_by_product.get(p_idx)
    pair_min_days[(k_idx, p_idx)] = (min(l for _, _, l in links) if links else DEFAULT_LEAD_DAYS) + 1

covered = random.sample(mix_pairs, len(mix_pairs) - RUPTURE_PAIRS)
low_pairs = set(random.sample(covered, int(len(covered) * LOW_STOCK_SHARE)))
batch_pairs = covered + random.choices(covered, k=ROWS_PER_TABLE - len(covered))
random.shuffle(batch_pairs)
seen_pairs = set()

rows = []
batches = []
for k_idx, p_idx in batch_pairs:
    product = products[p_idx]
    unit = product["unit"]
    daily = pair_daily_use[(k_idx, p_idx)]
    life = product["life"]
    perishable = life is not None and life[1] <= 60
    lot_life = random.randint(*life) if life else None
    coverage = random.uniform(3, 7) if perishable else random.uniform(8, 20)  # dias de consumo por entrega
    max_age = min(lot_life - 1 if lot_life else 90, 90)
    # o primeiro lote do par é o lote em uso; os demais são entregas anteriores, já consumidas
    newest = (k_idx, p_idx) not in seen_pairs
    seen_pairs.add((k_idx, p_idx))

    expired_chance = 0.08 if perishable else (0.03 if life and life[1] <= 200 else 0)
    roll = random.random()
    if roll < expired_chance:
        age = lot_life + random.randint(1, 6)
        initial = round_qty(daily * coverage, unit)
        current = round_qty(initial * random.uniform(0.1, 0.5), unit)
        status = "EXPIRED" if random.random() < 0.65 else "ACTIVE"
    elif roll < expired_chance + 0.015:
        # lançamento cancelado no recebimento
        age = random.randint(1, 30)
        initial, current, status = round_qty(daily * coverage, unit), 0, "CANCELLED"
    elif newest:
        min_days = pair_min_days[(k_idx, p_idx)]
        if (k_idx, p_idx) in low_pairs:
            remaining_days = min_days * random.uniform(0.2, 0.85)
        else:
            remaining_days = random.uniform(min_days * 1.2, min_days * 1.2 + coverage)
        age = random.randint(0, min(max_age, int(coverage)))
        current = round_qty(daily * remaining_days, unit)
        initial = round_qty(current + daily * age * random.uniform(0.9, 1.1), unit) if age else current
        status = "ACTIVE"
    else:
        age = random.randint(min(max_age, int(coverage) + 1), max_age)
        initial, current, status = round_qty(daily * coverage, unit), 0, "WRITTEN_OFF"

    entry_date = TODAY - timedelta(days=age)
    entry_dt = work_datetime(entry_date, 6, 15)
    expiration_date = entry_date + timedelta(days=lot_life) if lot_life else None

    links = links_by_product.get(p_idx, [])
    if links and random.random() < 0.97:
        supplier, reference_price, _ = random.choice(links)
        id_supplier, unit_price = supplier["id"], round(reference_price * random.uniform(0.97, 1.05), 2)
    else:
        # sem fornecedor: entrada por transferência entre unidades
        id_supplier, unit_price = None, round(product["price"] * random.uniform(0.95, 1.05), 2)

    batch_number = f"L{entry_date:%y%m%d}{random.choice('ABCDEFGH')}{random.randint(1, 9)}"
    invoice = random.randint(10_000, 999_999)
    invoice_number = f"{invoice // 1_000_000:03d}.{invoice // 1000 % 1000:03d}.{invoice % 1000:03d}" if id_supplier else None

    rows.append((
        product["id"], kitchens[k_idx]["id"], id_supplier, batch_number, invoice_number,
        initial, current, entry_date, expiration_date, unit_price, status,
    ))
    batches.append({
        "kitchen": k_idx, "product": p_idx, "unit": unit, "initial": initial, "current": current,
        "entry_dt": entry_dt, "expiration": expiration_date, "status": status, "number": batch_number,
        "daily": daily,
    })

result = execute_values(
    cursor,
    """INSERT INTO tb_stock_batch
        (id_product, id_kitchen, id_supplier, batch_number, invoice_number,
         initial_quantity, current_quantity, entry_date, expiration_date, unit_price, status)
       VALUES %s RETURNING id_batch""",
    rows,
    fetch=True,
)
for batch, (id_batch,) in zip(batches, result):
    batch["id"] = id_batch

conn.commit()
by_status = defaultdict(int)
for batch in batches:
    by_status[batch["status"]] += 1
print(f"{len(batches)} lotes de estoque criados: " + ", ".join(f"{n} {s}" for s, n in sorted(by_status.items())))

# 12. MOVIMENTAÇÕES DE ESTOQUE (tb_stock_movement)

O razão de cada lote é montado de ponta a ponta: entrada (`ENTRY`), baixas por uso, perda ou requisição aprovada (`WRITE_OFF`) e ajustes de inventário (`ADJUSTMENT`), com `balance_after` fechando exatamente no saldo atual do lote. As baixas de requisições de consumo e transferência aprovadas apontam para a requisição, como faz a API na aprovação.

A tabela guarda as 500 movimentações mais recentes desse razão (o histórico das últimas semanas).

In [ ]:
consumption_items = defaultdict(list)  # (cozinha, produto) -> itens de requisições aprovadas que baixam estoque
for req in requisitions:
    if req["status"] == "APPROVED" and req["type"] in ("CONSUMPTION", "TRANSFER"):
        for it in req["items"]:
            consumption_items[(req["kitchen"], it["product"])].append((req, it))

events = []
batch_events = {}
for b_idx, batch in enumerate(batches):
    if batch["status"] == "CANCELLED":
        batch_events[b_idx] = []
        continue
    k_idx, unit = batch["kitchen"], batch["unit"]
    step = to_milli(qty_step(unit))
    initial, current = to_milli(batch["initial"]), to_milli(batch["current"])
    stockkeeper = pick_staff(k_idx, ["estoquista", "supervisor"], batch["entry_dt"])
    lot = [{"type": "ENTRY", "at": batch["entry_dt"], "qty": initial, "reason": None, "user": stockkeeper, "req": None}]

    if current == 0:
        days_to_finish = batch["initial"] / max(batch["daily"], 0.001)
        end = min(NOW - timedelta(minutes=30), batch["entry_dt"] + timedelta(days=days_to_finish))
    elif batch["expiration"] and batch["expiration"] < TODAY:
        end = datetime.combine(batch["expiration"], time(18))
    else:
        end = NOW - timedelta(minutes=30)
    end = max(end, batch["entry_dt"] + timedelta(hours=2))

    adjustment = 0
    if current > 0 and random.random() < 0.06:
        adjustment = random.choice([-1, 1]) * max(step, initial * random.randint(1, 3) // 100 // step * step)
        # sobra (+) não pode passar do saldo atual e perda (-) não pode passar do que já saiu do lote
        if adjustment > current or initial - current + adjustment < 0:
            adjustment = 0
    to_write_off = initial - current + adjustment

    # baixas de requisição aprovada no período do lote
    for req, it in consumption_items.get((k_idx, batch["product"]), []):
        qty = to_milli(it["quantity"])
        if it.get("used") or not (batch["entry_dt"] < req["decided_at"] < end) or qty > to_write_off - step:
            continue
        it["used"] = True
        to_write_off -= qty
        lot.append({"type": "WRITE_OFF", "at": req["decided_at"], "qty": qty, "reason": "REQUISITION",
                    "user": req["approver"], "req": req["id"]})

    if to_write_off > 0:
        age_days = (end - batch["entry_dt"]).days
        for qty in split_milli(to_write_off, min(3, age_days // 5 + 1), step):
            reason = "LOSS" if random.random() < 0.06 else "USAGE"
            lot.append({"type": "WRITE_OFF", "at": rand_datetime(batch["entry_dt"] + timedelta(hours=1), end),
                        "qty": qty, "reason": reason, "user": pick_staff(k_idx, ["estoquista", "supervisor"]), "req": None})
    if adjustment:
        lot.append({"type": "ADJUSTMENT", "at": rand_datetime(batch["entry_dt"] + timedelta(hours=1), end),
                    "qty": adjustment, "reason": None, "user": pick_staff(k_idx, ["supervisor"]), "req": None})

    lot.sort(key=lambda e: e["at"])
    balance = 0
    for event in lot:
        balance += -event["qty"] if event["type"] == "WRITE_OFF" else event["qty"]
        if balance < 0:
            raise RuntimeError(f"Saldo negativo no lote {batch['id']}")
        event["balance"] = balance
        event["batch"] = b_idx
    if balance != current:
        raise RuntimeError(f"Razão do lote {batch['id']} fecha em {balance}, saldo é {current}")
    batch_events[b_idx] = lot
    events.extend(lot)

events.sort(key=lambda e: e["at"], reverse=True)
recent = sorted(events[:ROWS_PER_TABLE], key=lambda e: e["at"])

rows = [(
    batches[e["batch"]]["id"], e["user"]["id"] if e["user"] else None, e["req"], e["type"], e["reason"],
    from_milli(e["qty"]), from_milli(e["balance"]), e["at"],
) for e in recent]

execute_values(
    cursor,
    """INSERT INTO tb_stock_movement
        (id_batch, id_user, id_requisition, type, reason, quantity, balance_after, created_at)
       VALUES %s""",
    rows,
)

conn.commit()
by_type = defaultdict(int)
for e in recent:
    by_type[e["reason"] or e["type"]] += 1
print(f"{len(rows)} movimentações criadas (desde {recent[0]['at']:%d/%m/%Y}): " + ", ".join(f"{n} {t}" for t, n in sorted(by_type.items())))

# 13. PARÂMETROS DE ESTOQUE POR COZINHA (tb_product_kitchen_parameter)

Um parâmetro para cada par do mix. O consumo médio diário é o da cozinha; o estoque mínimo cobre o prazo de entrega do fornecedor mais um dia de segurança, e o máximo fica entre 2,5 e 4 vezes o mínimo.

In [ ]:
stock_by_pair = defaultdict(float)
for batch in batches:
    if batch["status"] == "ACTIVE" and (batch["expiration"] is None or batch["expiration"] >= TODAY):
        stock_by_pair[(batch["kitchen"], batch["product"])] += batch["current"]

rows = []
parameters = {}
for k_idx, p_idx in mix_pairs:
    product = products[p_idx]
    daily = pair_daily_use[(k_idx, p_idx)]
    min_stock = round_qty(daily * pair_min_days[(k_idx, p_idx)], product["unit"])
    max_stock = round_qty(min_stock * random.uniform(2.5, 4), product["unit"])
    parameters[(k_idx, p_idx)] = min_stock
    rows.append((product["id"], kitchens[k_idx]["id"], min_stock, max_stock, round(max(daily, 0.001), 3)))

execute_values(
    cursor,
    """INSERT INTO tb_product_kitchen_parameter
        (id_product, id_kitchen, min_stock, max_stock, average_daily_consumption)
       VALUES %s""",
    rows,
)

below_min = [pair for pair, min_stock in parameters.items() if stock_by_pair[pair] < min_stock]

conn.commit()
print(f"{len(rows)} parâmetros de estoque criados ({len(below_min)} pares abaixo do mínimo hoje)")

# 14. INVENTÁRIOS (tb_inventory)

Contagens cíclicas (rotativas) a cada dois dias nas últimas semanas: cada uma confere uma parte do estoque da cozinha. Em parte das cozinhas a contagem mais recente ainda está aberta.

In [ ]:
INVENTORY_NOTES = {
    "CLOSED": [
        "Inventário cíclico concluído sem pendências.",
        "Contagem da câmara fria de proteínas concluída.",
        "Conferência do estoque seco finalizada; divergências registradas.",
        "Inventário de hortifrúti concluído no fechamento do turno.",
        "Contagem rotativa de laticínios e frios concluída.",
    ],
    "OPEN": [
        "Contagem em andamento no estoque seco.",
        "Inventário cíclico iniciado no turno da manhã.",
        "Conferência da câmara de congelados em andamento.",
    ],
    "CANCELLED": [
        "Cancelado: equipe deslocada para o recebimento de mercadorias.",
        "Cancelado por queda de energia na câmara fria durante a contagem.",
        "Reagendado a pedido da supervisão.",
    ],
}

per_kitchen = [ROWS_PER_TABLE // len(kitchens)] * len(kitchens)
for k_idx in random.sample(range(len(kitchens)), ROWS_PER_TABLE - sum(per_kitchen)):
    per_kitchen[k_idx] += 1

inventories = []
rows = []
for k_idx, count in enumerate(per_kitchen):
    for n in range(count):
        # a cada dois dias: as contagens caem no período em que os lotes atuais já estavam no estoque
        day = TODAY - timedelta(days=2 * (count - 1 - n) + random.randint(0, 1))
        last = n == count - 1
        started_at = work_datetime(day, 7, 15)
        if last and random.random() < 0.5:
            status = "OPEN"
        else:
            status = "CANCELLED" if random.random() < 0.08 else "CLOSED"
        closed_at = started_at + timedelta(minutes=random.randint(90, 300)) if status == "CLOSED" else None
        if closed_at and closed_at > NOW:
            closed_at = NOW - timedelta(minutes=random.randint(1, 30))
        note = random.choice(INVENTORY_NOTES[status]) if status == "CANCELLED" or random.random() < 0.35 else None
        responsible = pick_staff(k_idx, ["estoquista", "supervisor"], started_at)

        rows.append((kitchens[k_idx]["id"], responsible["id"], started_at, closed_at, status, note))
        inventories.append({"kitchen": k_idx, "started_at": started_at, "status": status})

result = execute_values(
    cursor,
    """INSERT INTO tb_inventory (id_kitchen, id_responsible_user, started_at, closed_at, status, note)
       VALUES %s RETURNING id_inventory""",
    rows,
    fetch=True,
)
for inventory, (id_inventory,) in zip(inventories, result):
    inventory["id"] = id_inventory

conn.commit()
by_status = defaultdict(int)
for inventory in inventories:
    by_status[inventory["status"]] += 1
print(f"{len(inventories)} inventários criados: " + ", ".join(f"{n} {s}" for s, n in sorted(by_status.items())))

# 15. CONTAGENS DE INVENTÁRIO (tb_inventory_count)

Cada contagem confere um lote que estava em estoque no dia do inventário: a quantidade registrada é o saldo do lote naquele momento (calculado pelo razão da seção 12) e a física é o que a equipe contou. A maioria bate; as diferenças são pequenas e vêm com a justificativa. A coluna `divergence` é calculada pela trigger `trg_calculate_divergence`.

In [ ]:
DIVERGENCE_NOTES = [
    "Diferença atribuída a quebra no porcionamento.",
    "Embalagem avariada separada para descarte.",
    "Sobra de lote não baixado no turno anterior.",
    "Recontagem confirmou a diferença.",
    "Perda por descongelamento na câmara 2.",
]
MATCH_NOTES = ["Contagem conferida por dois estoquistas.", "Lote conferido com a etiqueta de recebimento."]


def balance_at(b_idx, moment):
    balance = 0
    for event in batch_events[b_idx]:
        if event["at"] > moment:
            break
        balance = event["balance"]
    return from_milli(balance)


batches_by_kitchen = defaultdict(list)
for b_idx, batch in enumerate(batches):
    batches_by_kitchen[batch["kitchen"]].append(b_idx)

eligible = {}
for i_idx, inventory in enumerate(inventories):
    if inventory["status"] == "CANCELLED":
        continue
    stocked = [b for b in batches_by_kitchen[inventory["kitchen"]] if balance_at(b, inventory["started_at"]) > 0]
    if stocked:
        random.shuffle(stocked)
        eligible[i_idx] = stocked

slots = [(i, 0) for i in eligible]
depth = 1
while len(slots) < ROWS_PER_TABLE:
    deeper = [(i, depth) for i, stocked in eligible.items() if len(stocked) > depth]
    if not deeper:
        raise RuntimeError(f"Só há {len(slots)} contagens possíveis; o esperado é {ROWS_PER_TABLE}")
    slots += deeper
    depth += 1
slots = slots[:len(eligible)] + random.sample(slots[len(eligible):], ROWS_PER_TABLE - len(eligible))

rows = []
for i_idx, position in slots:
    inventory = inventories[i_idx]
    b_idx = eligible[i_idx][position]
    unit = batches[b_idx]["unit"]
    registered = balance_at(b_idx, inventory["started_at"])
    roll = random.random()
    if roll < 0.72:
        physical = registered
    elif roll < 0.92:
        physical = max(0, registered - round_qty(registered * random.uniform(0.005, 0.04), unit))
    else:
        physical = registered + round_qty(registered * random.uniform(0.005, 0.02), unit)
    physical = round(physical, 3)
    if physical != registered:
        note = random.choice(DIVERGENCE_NOTES) if random.random() < 0.7 else None
    else:
        note = random.choice(MATCH_NOTES) if random.random() < 0.08 else None
    rows.append((inventory["id"], batches[b_idx]["id"], registered, physical, note))

execute_values(
    cursor,
    """INSERT INTO tb_inventory_count (id_inventory, id_batch, registered_quantity, physical_quantity, note)
       VALUES %s""",
    rows,
)

conn.commit()
divergent = sum(1 for r in rows if r[2] != r[3])
print(f"{len(rows)} contagens de inventário criadas ({divergent} com divergência)")

# 16. ALERTAS (tb_alert)

Os alertas são os que o sistema geraria a partir dos dados acima, com os tipos, severidades e mensagens da API e do banco:

- **Abertos hoje:** lotes vencidos (`EXPIRATION`, já criados pela trigger na seção 11), produtos abaixo do estoque mínimo (`STOCK`), lotes que vencem nos próximos 3 dias (`EXPIRING`), requisições aguardando o supervisor (`REQUISITION_SUBMITTED`), decisões recentes para o solicitante (`REQUISITION_APPROVED` / `REQUISITION_REJECTED`), pedidos de entrada na cozinha (`ACCESS_REQUEST`) e produtos escaneados fora do catálogo (`UNREGISTERED_PRODUCT`).
- **Histórico já tratado:** alertas antigos desses mesmos tipos, resolvidos (`is_read = true`) ou lidos pelo destinatário.

O total fecha em exatamente 500, somando os alertas da trigger.

In [ ]:
UNREGISTERED_NAMES = [
    "Molho Barbecue Heinz 2 kg", "Queijo Gorgonzola Tirolez", "Palmito Pupunha em Conserva 1,8 kg",
    "Azeitona Verde sem Caroço 2 kg", "Cogumelo Champignon Fatiado 2 kg", "Batata Palha Extrafina 1 kg",
    "Creme Culinário 1 kg", "Tapioca Granulada 500 g", "Molho Pesto 1 kg", "Requeijão Culinário 1,8 kg",
]

cursor.execute("SELECT COUNT(*) FROM tb_alert")
trigger_alerts = cursor.fetchone()[0]
supervisor_profile = profile_id_by_code["supervisor"]


def alert(kind, severity, k_idx, message, created_at, is_read, b_idx=None, p_idx=None,
          user=None, target_supervisor=False, ref_key=None):
    return {
        "type": kind, "severity": severity, "kitchen": k_idx, "message": message, "created_at": created_at,
        "is_read": is_read, "batch": b_idx, "product": p_idx, "user": user,
        "target": supervisor_profile if target_supervisor else None, "ref_key": ref_key,
    }


def expiring_message(batch, moment_day):
    days = (batch["expiration"] - moment_day).days
    when = "vence hoje" if days == 0 else ("vence amanhã" if days == 1 else f"vence em {days} dias")
    return f"Lote {batch['number']} {when} ({batch['expiration']:%d/%m})."


def job_time(day):
    """Horário do job diário de validade (logo depois da meia-noite)."""
    return min(datetime.combine(day, time(0, 5)) + timedelta(seconds=random.randint(0, 600)), NOW)


open_alerts, history = [], defaultdict(list)

# estoque abaixo do mínimo (alerta da cozinha inteira)
for k_idx, p_idx in below_min:
    open_alerts.append(alert("STOCK", "HIGH", k_idx, "Produto abaixo do estoque mínimo.",
                             NOW - timedelta(minutes=random.randint(30, 72 * 60)), False, p_idx=p_idx))

# validade: vence nos próximos 3 dias (aberto) e avisos que antecederam os lotes já vencidos (resolvidos)
for b_idx, batch in enumerate(batches):
    if batch["expiration"] is None:
        continue
    if batch["status"] == "ACTIVE" and TODAY <= batch["expiration"] <= TODAY + timedelta(days=3):
        days = (batch["expiration"] - TODAY).days
        open_alerts.append(alert("EXPIRING", "HIGH" if days <= 1 else "MEDIUM", batch["kitchen"],
                                 expiring_message(batch, TODAY), job_time(TODAY), False, b_idx, batch["product"]))
    elif batch["expiration"] < TODAY:
        warn_day = batch["expiration"] - timedelta(days=3)
        if warn_day > batch["entry_dt"].date():
            history["EXPIRING"].append(alert("EXPIRING", "MEDIUM", batch["kitchen"], expiring_message(batch, warn_day),
                                             job_time(warn_day), True, b_idx, batch["product"]))

# requisições: aviso ao supervisor no envio e ao solicitante na decisão
for req in requisitions:
    if req["status"] == "DRAFT" or (req["status"] == "CANCELLED" and random.random() < 0.5):
        continue
    submitted_at = min(req["created_at"] + timedelta(minutes=random.randint(2, 90)), NOW - timedelta(minutes=1))
    submitted = alert("REQUISITION_SUBMITTED", "MEDIUM", req["kitchen"], f"Nova requisição #{req['id']} para análise.",
                      submitted_at, req["status"] != "UNDER_REVIEW", target_supervisor=True, ref_key=str(req["id"]))
    (open_alerts if req["status"] == "UNDER_REVIEW" else history["REQUISITION_SUBMITTED"]).append(submitted)

    if req["status"] in ("APPROVED", "REJECTED"):
        if req["status"] == "APPROVED":
            kind, message = "REQUISITION_APPROVED", f"Sua requisição #{req['id']} foi aprovada."
        else:
            kind, message = "REQUISITION_REJECTED", f"Sua requisição #{req['id']} foi recusada: {req['reason']}"
        decision = alert(kind, "MEDIUM", req["kitchen"], message, req["decided_at"], False,
                         user=req["requester"], ref_key=str(req["id"]))
        (open_alerts if req["decided_at"] > NOW - timedelta(days=2) else history["DECISION"]).append(decision)

# pedidos de entrada na cozinha: os 20 usuários sem cozinha aguardam; os antigos já foram aceitos
for user in users:
    if user["kitchen"] is None:
        k_idx = random.choices(range(len(kitchens)), weights=kitchen_weights)[0]
        open_alerts.append(alert("ACCESS_REQUEST", "MEDIUM", k_idx, f"{user['name']} pediu para entrar na cozinha.",
                                 min(user["created_at"] + timedelta(minutes=random.randint(5, 120)), NOW),
                                 False, target_supervisor=True, ref_key=fake.uuid4()))
    elif user["created_at"] > NOW - timedelta(days=365):
        history["ACCESS_REQUEST"].append(alert(
            "ACCESS_REQUEST", "MEDIUM", user["kitchen"], f"{user['name']} pediu para entrar na cozinha.",
            user["created_at"] - timedelta(minutes=random.randint(60, 30 * 60)), True,
            target_supervisor=True, ref_key=fake.uuid4()))

# produto escaneado fora do catálogo: abertos (ainda não cadastrados) e históricos (cadastrados depois)
for name in random.sample(UNREGISTERED_NAMES, 6):
    barcode = ean13()
    while barcode in used_barcodes:
        barcode = ean13()
    used_barcodes.add(barcode)
    open_alerts.append(alert("UNREGISTERED_PRODUCT", "MEDIUM", random.randrange(len(kitchens)),
                             f"Produto não cadastrado: {name} ({barcode})", past_work_datetime(0, 5), False,
                             target_supervisor=True, ref_key=barcode))
mix_kitchens = defaultdict(list)
for k_idx, p_idx in mix_pairs:
    mix_kitchens[p_idx].append(k_idx)
for p_idx, product in enumerate(products):
    if product["barcode"] and product["created_at"] > NOW - timedelta(days=150):
        k_idx = random.choice(mix_kitchens[p_idx]) if mix_kitchens[p_idx] else random.randrange(len(kitchens))
        history["UNREGISTERED_PRODUCT"].append(alert(
            "UNREGISTERED_PRODUCT", "MEDIUM", k_idx, f"Produto não cadastrado: {product['name']} ({product['barcode']})",
            product["created_at"] - timedelta(minutes=random.randint(120, 3 * 24 * 60)), True,
            target_supervisor=True, ref_key=product["barcode"]))

# estoque mínimo já reposto
for k_idx, p_idx in random.sample(mix_pairs, 120):
    history["STOCK"].append(alert("STOCK", "HIGH", k_idx, "Produto abaixo do estoque mínimo.",
                                  NOW - timedelta(minutes=random.randint(5 * 24 * 60, 80 * 24 * 60)), True, p_idx=p_idx))

room = ROWS_PER_TABLE - trigger_alerts - len(open_alerts)
if room < 0:
    raise RuntimeError(f"{trigger_alerts + len(open_alerts)} alertas abertos passam de {ROWS_PER_TABLE}")

# o histórico completa os 500, na proporção abaixo
HISTORY_SHARE = {"DECISION": 0.38, "REQUISITION_SUBMITTED": 0.30, "STOCK": 0.12, "ACCESS_REQUEST": 0.08,
                 "UNREGISTERED_PRODUCT": 0.06, "EXPIRING": 0.06}
picked = []
for kind, share in HISTORY_SHARE.items():
    random.shuffle(history[kind])
    picked += history[kind][:min(len(history[kind]), int(room * share))]
leftover = [a for kind in HISTORY_SHARE for a in history[kind] if a not in picked]
picked += random.sample(leftover, room - len(picked))

alerts = sorted(open_alerts + picked, key=lambda a: a["created_at"])
rows = [(
    a["type"], a["severity"], batches[a["batch"]]["id"] if a["batch"] is not None else None,
    products[a["product"]]["id"] if a["product"] is not None else None, kitchens[a["kitchen"]]["id"],
    a["message"], a["is_read"], a["created_at"], a["user"]["id"] if a["user"] else None, a["target"], a["ref_key"],
) for a in alerts]

result = execute_values(
    cursor,
    """INSERT INTO tb_alert
        (type, severity, id_batch, id_product, id_kitchen, message, is_read, created_at, id_user, id_target_profile, ref_key)
       VALUES %s RETURNING id_alert""",
    rows,
    fetch=True,
)
for a, (id_alert,) in zip(alerts, result):
    a["id"] = id_alert
open_ids = {id(a) for a in open_alerts}

conn.commit()
print(f"{len(alerts) + trigger_alerts} alertas no total: {trigger_alerts} de lote vencido (trigger), "
      f"{len(open_alerts)} abertos gerados aqui e {len(picked)} do histórico")

# 17. LEITURAS DE ALERTA (tb_alert_read)

Quem leu cada alerta do histórico: o solicitante lê a decisão da sua requisição, os supervisores leem os avisos dirigidos a eles e a equipe da cozinha lê os alertas gerais. Alertas abertos não têm leitura (é o que os mantém em `vw_active_alerts`).

In [ ]:
kitchen_staff = defaultdict(list)
for (k_idx, _profile), members in staff.items():
    kitchen_staff[k_idx] += members


def readers_of(a):
    if a["user"]:
        return [a["user"]]
    if a["target"]:
        return list(staff[(a["kitchen"], "supervisor")])
    return list(kitchen_staff[a["kitchen"]])


first_reads, extra_reads = [], []
for a in alerts:
    if id(a) in open_ids:
        continue
    readers = [u for u in readers_of(a) if u["created_at"] < a["created_at"]]
    random.shuffle(readers)
    if readers:
        first_reads.append((a, readers[0]))
        extra_reads += [(a, u) for u in readers[1:8]]

if len(first_reads) + len(extra_reads) < ROWS_PER_TABLE:
    raise RuntimeError(f"Só há {len(first_reads) + len(extra_reads)} leituras possíveis")
chosen = first_reads[:ROWS_PER_TABLE]
chosen += random.sample(extra_reads, ROWS_PER_TABLE - len(chosen))

rows = []
for a, reader in chosen:
    read_at = min(a["created_at"] + timedelta(minutes=random.randint(4, 26 * 60)), NOW - timedelta(minutes=1))
    rows.append((a["id"], reader["id"], read_at))

execute_values(cursor, "INSERT INTO tb_alert_read (id_alert, id_user, read_at) VALUES %s", rows)

conn.commit()
print(f"{len(rows)} leituras de alerta registradas")

# 18. ACESSOS DIÁRIOS (tb_user_daily_access)

Base do DAU (Daily Active Users). Em produção, esta tabela é gravada só pela trigger `trg_register_daily_access` quando a API atualiza `tb_user.last_login`. O seed insere direto o histórico das últimas 3 semanas, porque a trigger só registra o login "de agora". O último acesso de cada usuário é exatamente o `last_login` gravado na seção 7, e os demais dias completam as 500 linhas (dias úteis com mais acessos que fins de semana).

In [ ]:
window_start = TODAY - timedelta(days=HISTORY_DAYS - 1)

mandatory, optional = [], []
for user in users:
    last_login = user["last_login"]
    if last_login is None or last_login.date() < window_start:
        continue
    last_day = last_login.date()
    count = random.choices([1, 2, 3], weights=[55, 30, 15])[0]
    first = last_login
    if count > 1:
        first = max(datetime.combine(last_day, time(0)), last_login - timedelta(minutes=random.randint(30, 480)))
    mandatory.append((user["id"], last_day, first.time(), last_login.time(), count))

    heavy_user = random.uniform(0.3, 1.0)
    day = max(window_start, user["created_at"].date() + timedelta(days=1))
    while day < last_day:
        weight = heavy_user * (1 if day.weekday() < 5 else 0.5)
        optional.append((random.random() ** (1 / weight), user["id"], day))
        day += timedelta(days=1)

if len(mandatory) > ROWS_PER_TABLE:
    mandatory = random.sample(mandatory, ROWS_PER_TABLE)
optional.sort(reverse=True)

rows = list(mandatory)
for _, id_user, day in optional[:ROWS_PER_TABLE - len(rows)]:
    count = random.choices([1, 2, 3, 4], weights=[50, 30, 15, 5])[0]
    first = datetime.combine(day, time(5, 30)) + timedelta(minutes=random.randint(0, 270))
    last = min(first + timedelta(minutes=random.randint(30, 600)), datetime.combine(day, time(23, 59))) if count > 1 else first
    rows.append((id_user, day, first.time(), last.time(), count))

execute_values(
    cursor,
    """INSERT INTO tb_user_daily_access
        (id_user, access_date, first_access_time, last_access_time, access_count)
       VALUES %s""",
    rows,
)

conn.commit()
print(f"{len(rows)} acessos diários criados ({HISTORY_DAYS} dias de histórico)")

# RELATÓRIO EXECUTIVO J&F

In [ ]:
EXPECTED_ROWS = {
    "tb_profile": len(PROFILES), "tb_kitchen": len(KITCHENS), "tb_category": len(CATEGORIES) + len(group_category_id),
    "tb_measurement_unit": len(UNITS), "tb_supplier": ROWS_PER_TABLE, "tb_product": ROWS_PER_TABLE,
    "tb_user": ROWS_PER_TABLE, "tb_product_supplier": ROWS_PER_TABLE, "tb_requisition": ROWS_PER_TABLE,
    "tb_requisition_item": ROWS_PER_TABLE, "tb_stock_batch": ROWS_PER_TABLE, "tb_stock_movement": ROWS_PER_TABLE,
    "tb_product_kitchen_parameter": ROWS_PER_TABLE, "tb_inventory": ROWS_PER_TABLE,
    "tb_inventory_count": ROWS_PER_TABLE, "tb_alert": ROWS_PER_TABLE, "tb_alert_read": ROWS_PER_TABLE,
    "tb_user_daily_access": ROWS_PER_TABLE,
}
LOG_TABLES = [
    "tb_log_user", "tb_log_product", "tb_log_supplier", "tb_log_stock_batch",
    "tb_log_requisition", "tb_log_inventory", "tb_log_alert",
]

print("\n" + "=" * 70)
print("     RELATÓRIO EXECUTIVO - INVENTRA (GRUPO J&F)")
print("=" * 70)

cursor.execute("""
    SELECT k.name, SUM(sb.current_quantity * COALESCE(sb.unit_price, 0)) AS valor
    FROM tb_stock_batch sb
    JOIN tb_kitchen k ON k.id_kitchen = sb.id_kitchen
    WHERE sb.status = 'ACTIVE'
    GROUP BY k.name
    ORDER BY valor DESC
    LIMIT 5
""")
print("\nTOP 5 COZINHAS POR VALOR EM ESTOQUE:")
for nome, valor in cursor.fetchall():
    print(f"   {nome:<48} R$ {valor:>12,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

cursor.execute("""
    SELECT COALESCE(t.category_path, 'Sem categoria'), COUNT(*)
    FROM tb_product p
    LEFT JOIN vw_category_tree t ON t.id_category = p.id_category
    GROUP BY 1
    ORDER BY 2 DESC
""")
print("\nPRODUTOS POR CATEGORIA:")
for categoria, qtd in cursor.fetchall():
    print(f"   {categoria:<55} {qtd}")

cursor.execute("SELECT name FROM tb_product WHERE photo_url IS NULL ORDER BY random() LIMIT 4")
print("\nAMOSTRA DO CATÁLOGO CURADO:")
for (nome,) in cursor.fetchall():
    print(f"   • {nome}")

cursor.execute("SELECT COUNT(*) FROM tb_product WHERE photo_url IS NOT NULL")
print(f"\nPRODUTOS REAIS DA OPEN FOOD FACTS (com foto): {cursor.fetchone()[0]}")
cursor.execute("SELECT name, barcode, photo_url FROM tb_product WHERE photo_url IS NOT NULL ORDER BY random() LIMIT 4")
for nome, barcode, photo_url in cursor.fetchall():
    print(f"   • {nome} ({barcode})\n     {photo_url}")

cursor.execute("SELECT status, COUNT(*) FROM tb_stock_batch GROUP BY status ORDER BY COUNT(*) DESC")
print("\nLOTES DE ESTOQUE POR STATUS:")
for status, qtd in cursor.fetchall():
    print(f"   {status:<15} {qtd}")

cursor.execute("SELECT status, COUNT(*) FROM tb_requisition GROUP BY status ORDER BY COUNT(*) DESC")
print("\nREQUISIÇÕES POR STATUS:")
for status, qtd in cursor.fetchall():
    print(f"   {status:<15} {qtd}")

cursor.execute("SELECT type, COUNT(*) FROM vw_active_alerts GROUP BY type ORDER BY COUNT(*) DESC")
print("\nALERTAS ABERTOS POR TIPO (vw_active_alerts):")
for tipo, qtd in cursor.fetchall():
    print(f"   {tipo:<25} {qtd}")

cursor.execute("""
    SELECT access_date, dau, dau_moving_avg_7d, mau_30d, stickiness_pct
    FROM vw_dau
    ORDER BY access_date DESC
    LIMIT 7
""")
print("\nDAU DOS ÚLTIMOS 7 DIAS (vw_dau):")
print(f"   {'dia':<12} {'DAU':>5} {'média 7d':>9} {'MAU 30d':>8} {'DAU/MAU':>8}")
for dia, dau, media, mau, stickiness in cursor.fetchall():
    print(f"   {dia!s:<12} {dau:>5} {media:>9} {mau:>8} {stickiness!s:>7}%")

print("\n" + "=" * 70)
print("     RESUMO FINAL - LINHAS POR TABELA")
print("=" * 70)
total_geral = 0
for table, expected_rows in EXPECTED_ROWS.items():
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    qtd = cursor.fetchone()[0]
    total_geral += qtd
    flag = "OK" if qtd == expected_rows else f"ESPERADO {expected_rows}"
    print(f"   {table:<32} {qtd:>6}  [{flag}]")

print("\n   Tabelas de log (preenchidas via trigger, sem INSERT direto):")
for table in LOG_TABLES:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"   {table:<32} {cursor.fetchone()[0]:>6}")

print(f"\nTotal de linhas nas {len(EXPECTED_ROWS)} tabelas de negócio: {total_geral}")
print("=" * 70)

# VERIFICAÇÃO (NOVA CONEXÃO)

Abre uma segunda conexão, que só enxerga dados já gravados (commitados). Se aqui as tabelas aparecerem preenchidas, o seed gravou; para vê-las no pgAdmin, abra exatamente o banco indicado abaixo.

In [ ]:
check = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE", "require"),
)
check_cursor = check.cursor()
check_cursor.execute("SELECT current_database(), current_user")
db_name, db_user = check_cursor.fetchone()
print(f"Verificação em {db_user}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}  ->  banco '{db_name}' (nova conexão)")
print("No pgAdmin: Servers > (servidor) > Databases > " + db_name + " > Schemas > public > Tables\n")

for table in EXPECTED_ROWS:
    check_cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"   {table:<32} {check_cursor.fetchone()[0]:>6}")

check_cursor.close()
check.close()

# FINALIZAR

In [ ]:
cursor.close()
conn.close()

print("\nSeed Inventra (J&F) concluído com sucesso!")
print("Banco pronto para o sistema de gestão de estoque e cozinhas industriais")
print("\nDestaques da carga:")
print("   • 40 cozinhas industriais de Friboi, Seara, Swift e JBS em cidades onde o grupo opera")
print("   • 500 linhas em cada tabela de cadastro e movimento; domínios (perfis, categorias, unidades) no tamanho real")
print("   • Produtos reais da Open Food Facts (nome, EAN e foto) somados a um catálogo curado de cozinha industrial")
print("   • Saldo de cada lote fecha com o razão de movimentações; alertas refletem o estoque e as requisições")
print("   • Inserts em lote via execute_values (poucas viagens de rede por tabela, não uma por linha)")
print("   • Tabelas tb_log_* preenchidas automaticamente pelas triggers de auditoria")

# ROLLBACK (OPCIONAL) — APAGAR DADOS DO SEED

In [ ]:
# Célula independente: não depende de nenhuma célula acima ter rodado nesta
# sessão do kernel. Roda ela sozinha sempre que quiser desfazer a carga
# deste seed — abre e fecha sua própria conexão (a célula FINALIZAR já
# fechou a conexão usada pelo resto do notebook).
#
# ATENÇÃO: destrutivo. Apaga TODAS as linhas das tabelas abaixo, não só as
# inseridas por este seed. RESTART IDENTITY zera os contadores de novo.
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

rollback_conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE", "require"),
)
rollback_cursor = rollback_conn.cursor()

rollback_cursor.execute("""
TRUNCATE TABLE
    tb_profile,
    tb_kitchen,
    tb_category,
    tb_measurement_unit,
    tb_user,
    tb_product,
    tb_supplier,
    tb_product_supplier,
    tb_product_kitchen_parameter,
    tb_stock_batch,
    tb_stock_movement,
    tb_requisition,
    tb_requisition_item,
    tb_inventory,
    tb_inventory_count,
    tb_alert,
    tb_alert_read,
    tb_user_daily_access
RESTART IDENTITY CASCADE;
""")
rollback_cursor.execute("""
TRUNCATE TABLE
    tb_log_user,
    tb_log_product,
    tb_log_supplier,
    tb_log_stock_batch,
    tb_log_requisition,
    tb_log_inventory,
    tb_log_alert
RESTART IDENTITY;
""")
rollback_conn.commit()

rollback_cursor.close()
rollback_conn.close()

print("Rollback concluído: todos os dados inseridos pelo seed foram removidos.")